## <b>METADATA</b>
***

<b>Project</b>: Universal Declaration of Human Rights on DNAmaker - 2025
<br><b>Protocol</b>: Construction of BigBlocks by Golden Gate Assembly and PCR selection</br>
<br><b>Description</b>:</br>
<BLOCKQUOTE><div style="margin: 5px; line-height: 1.5;">- <font style="color:#EB5406;"><font style="line-height:20px;"> <b>Part 1. Build 12 BigBlocks with 8 eBlocks with studies of the reproducibility of enzymatic reactions</b>: <font style="color:black;">one master-mix for 12 BigBlocks, subdivise in 12 mini master-mix, build one BigBlock per mini master-mix.</div></BLOCKQUOTE>
    
<BLOCKQUOTE><div style="margin: 5px; line-height: 1.5;">- <font style="color:#EB5406;"><font style="line-height:20px;"> <b>Part 2. PCR on the 12 BigBlocks</b>: <font style="color:black;">one master-mix, subdivise in 12 mini master-mix, add one BigBlock per mini master-mix.</div></BLOCKQUOTE>


<b>Author</b>: Julien Leblanc - julien.leblanc@irisa.fr |
<b>Version</b>: Last updated on <b>May 06, 2025</b> |
<b>ApiLevel</b>: 2.19 |
<b>Execution</b>: on local OT-2 server through Jupyter Notebook |
<b>Simulation</b>: first launch Anaconda, next load Python 3.10 env with Opentrons lib and launch Jupyter Notebook

## **IMPORT & SET DOBOT and FRIDGE**

In [1]:
import sys
sys.path.insert(0,'/var/user-packages/usr/lib/python3.10/site-packages/')

In [ ]:
# for fridge 
from dynio import *
import time
class Fridge:
    
    open_angle = 105
    close_angle = 10
    middle_angle = 50
    angle_tolerance = 0.5
    low_angle_speed = 20
    high_angle_speed = 50
    control_period = 0.01
    def __init__(self,serial_path="/dev/ttyACM0",motor_id=1):
      self.dxl_io = dxl.DynamixelIO(serial_path)
      self.mx_64 = self.dxl_io.new_mx64(motor_id, 1)  
    def _set_angle(self,target_angle):
      while 1:
        angle = self.mx_64.get_angle()
        if abs(angle - target_angle) < self.angle_tolerance:
          break
        if angle < self.middle_angle:
          self.mx_64.set_velocity(self.high_angle_speed)
        else:
          self.mx_64.set_velocity(self.low_angle_speed)
        self.mx_64.set_angle(target_angle)
        self.mx_64.write_control_table("P_Gain", 32)
        self.mx_64.write_control_table("D_Gain", 20)
        self.mx_64.torque_enable()
        time.sleep(self.control_period)
    def open_door(self):
      self._set_angle(self.open_angle)
    def close_door(self):
      self._set_angle(self.close_angle)
    def release_door(self):
      self.mx_64.write_control_table("P_Gain", 0)
      self.mx_64.write_control_table("D_Gain", 0)
      self.mx_64.set_angle(self.mx_64.get_angle())
      self.mx_64.torque_disable()
    def is_door_open(self):
        angle = self.mx_64.get_angle()
        if abs(self.open_angle - angle) < 2*self.angle_tolerance:
            return True
    def is_door_close(self):
        angle = self.mx_64.get_angle()
        if abs(self.close_angle - angle) < 2*self.angle_tolerance:
            return True

In [ ]:
import threading
from dobot_api import DobotApiDashboard, DobotApi, DobotApiMove, MyType,alarmAlarmJsonFile
from types import SimpleNamespace
import requests
import time
import tempfile
import urllib
from smb.SMBHandler import SMBHandler

class Dobot:
    slide_length = 800 
    ip = "169.254.111.111" # configured IP for LAN2 
    ip_debug_lan1 = "192.168.1.6" # debug ip fixed on LAN 1
    dashboardPort = 29999
    movePort = 30003
    feedPort = 30004
    web_port = 22000
    rail_offset = 0
    rail_acc = 1
    rail_speed = 1
    def change_lan2_ip(self, new_ip):
        payload = {    
        "dhcp": False,
        "gateway": new_ip,
        "ip": new_ip,
        "netmask": "255.255.0.0"
        }
        res = requests.post(
            f"http://{self.ip_debug_lan1}:{self.web_port}/interface/ethernet",
            json=payload)
        self.ip = new_ip
    
    def connect(self):
        self.dashboard = DobotApiDashboard(self.ip, self.dashboardPort)
        self.move = DobotApiMove(self.ip, self.movePort)
        self.feed = DobotApi(self.ip, self.feedPort)
        self.enable_rail(True)
    def get_pose(self):
        while True:
            res = self.dashboard.GetPose()
            if res.split(',')[-1] =='GetPose();':
                return [float(i) for i in res.replace('{','').replace('}','').split(',')[1:7]]
    def open_gripper(self): 
        self.dashboard.DO(1,0)
        self.dashboard.DO(2,0)
        time.sleep(0.3)
    def close_gripper(self):
        self.dashboard.DO(1,1)
        self.dashboard.DO(2,0)
        time.sleep(0.3)
    def get_robot_status(self):
        modesDict={
            "1" : ("ROBOT_MODE_INIT", "initialization"),
            "2" :	("ROBOT_MODE_BRAKE_OPEN","The brake release"),
            "3" :	("","Reserved"),
            "4" :	("ROBOT_MODE_DISABLED","Disabled (brake is not released)"),
            "5" :	("ROBOT_MODE_ENABLE","Enable (idle)"),
            "6" :	("ROBOT_MODE_BACKDRIVE","Dragging statej"),
            "7" :	("ROBOT_MODE_RUNNING","Running state"),
            "8" :	("ROBOT_MODE_RECORDING","Dragging recording"),
            "9" :	("ROBOT_MODE_ERROR","Alarm state"),
            "10" : ("ROBOT_MODE_PAUSE","pause state"),
            "11" : ("ROBOT_MODE_JOG","jogging state")
        }
        return modesDict[self.dashboard.RobotMode().split('{')[1].split('}')[0]]
    def _post(self,path,json_data=None):
        res = requests.post(
            f'http://{self.ip}:{self.web_port}/{path}',
            json=json_data)
        assert(res.status_code == 200)
        return res.json()
    def _get(self, path):
        res = requests.get(f'http://{self.ip}:{self.web_port}/{path}')
        assert(res.status_code == 200)
        return res.json()
    def set_global_speed_ratio(self, ratio=100):
        
        self.dashboard.SpeedFactor(ratio)

        # need to write a file on DOBOT using SMB protocol
        target_smb_url = f'smb://MG_400,{self.ip}/project/settings/common.json'
        data = {"ratio": ratio}
        tmp = tempfile.NamedTemporaryFile()
        with open(tmp.name, 'w') as f:
            f.write(f"""{{
    "ratio": {ratio}
}}
""")
        opener = urllib.request.build_opener(SMBHandler)

        with open(tmp.name, 'rb') as f:
            with opener.open(target_smb_url, data=f) as fh:
                print("write ratio to DOBOT")
        return self._post("settings/common", data)
    
    def get_emergency_stop_state(self):
        return self._get("panel/emergencyStop")
    def get_load_params(self):
        return self._get("settings/function/loadParams")
    def set_load_params(self, data):
        """ data = {
    "inertiaX": 0.10000000,
    "inertiaY": 0.10000000,
    "inertiaZ": 0,
    "loadValue": 0.2
}"""
        # need to write a file on DOBOT using SMB protocol
        target_smb_url = f'smb://MG_400,{self.ip}/project/settings/function/loadParams.json'
        tmp = tempfile.NamedTemporaryFile()
        with open(tmp.name, 'w') as f:
            f.write(f"""{{
    "inertiaX": {data["inertiaX"]},
    "inertiaY": {data["inertiaY"]},
    "inertiaZ": {data["inertiaZ"]},
    "loadValue": {data["loadValue"]}
}}""")
        opener = urllib.request.build_opener(SMBHandler)

        with open(tmp.name, 'rb') as f:
            with opener.open(target_smb_url, data=f) as fh:
                print("write data to DOBOT")
        return self._post("settings/common", data)
        
        return self._post("settings/function/loadParams",data)
    def get_user_param(self):
        return self._get("settings/driver/userParam")
    def get_connexion_state(self):
        return self._get("connection/state")
    def set_control_mode(self, data):
        """ data = {
    "controlMode": "enable"
    }
"""
        return self._post("settings/controlMode",data)
    

    def print_error_message(self):
        import json
        from files.alarm_controller import alarm_controller_list
        from files.alarm_servo import alarm_servo_list
        def convert_dict(alarm_list):
            alarm_dict = {}
            for i in alarm_list:
                alarm_dict[i["id"]] = i
            return alarm_dict
        def get_error(index, alarm_dict: dict, type_text):
            resp =  f"""ID:{index}
            Type:{type_text}"""
            if index in alarm_dict :
                resp += f"""
                Level:{alarm_dict[index]['level']}
                Solution:{alarm_dict[index]['en']['solution']}
            """
            else:
                resp+= f"""
                Level: Null
                Solution: Null
                """
                return resp
        alarm_controller_dict = convert_dict(alarm_controller_list)
        alarm_servo_dict = convert_dict(alarm_servo_list)
        error_list = json.loads(self.dashboard.GetErrorID().split("{")[1].split("}")[0])
        if error_list[0]:
            for i in error_list[0]:
                print(get_error(i,alarm_controller_dict, "Controller Error"))
        for m in range(1, len(error_list)):
            if error_list[m]:
                for n in range(len(error_list[m])):
                    print(get_error(n, alarm_servo_dict, "Servo Error"))

    def enable_rail(self,value=True):
        return self._post(
            "process/auxJoint/switch",
            {'auxJoint': value, 'addPoints': False}
            )
    def rail(self,value):
        return self.slide_length - value
        
    def calibrate_rail(self,true_pos=800):
        res = requests.post(
            f'http://{self.ip}:{self.web_port}/calibrate/homeAuxJoint')
        self._post(
            "calibrate/homeAuxJoint",
            None
            )
        data = self._get("protocol/exchange")
        actual_pos =self.rail(data["auxJoint"][0])
        self.rail_offset = true_pos - actual_pos
        
    def mov_arm_joints_and_rail(self,j1,j2,j3,j4,j_ext):
        rail_j = j_ext - self.rail_offset
        self.move.JointMovJ(j1,j2,j3,j4)
        self.move.MovJExt(self.rail(rail_j),self.rail_speed,self.rail_acc,1) # SpeedE=5 /100, AccE=5 / 100, SYNC=1 

    def mov_arm_coords_and_rail(self,x,y,z,r,y_ext,sync=True):
        rail_y = y_ext - self.rail_offset
        self.move.MovJ(x,y,z,r,) # {CP=1 [0:100], SpeedJ=50 [0:100], AccJ=20 [0:100]}
        self.move.MovJExt(self.rail(rail_y),self.rail_speed,self.rail_acc,1) # SpeedE=5 [0:100], AccE=5 [0:100], SYNC=1 [0,1]
        if sync:
            while not self.is_close_to(x,y,z,r,y_ext) :
                time.sleep(0.1)
        
    def is_close_to(self,x,y,z,r,y_ext):
        current_x,current_y,current_z,current_r,current_y_ext = self.get_arm_and_rail_coordinates()
        if abs(current_x-x)+abs(current_y-y)+abs(current_z-z)+abs(current_y_ext-y_ext)<2.0: 
            return True
        else:
            return False
    def get_arm_and_rail_joint_values(self):
        data= self._get("protocol/exchange")
        rail_j = self.rail(data["auxJoint"][0])
        j_ext = rail_j + self.rail_offset
        return data["jointCoordinate"]+[j_ext]
    
    def get_arm_and_rail_coordinates(self):
        data= self._get("protocol/exchange")
        x,y,z,r = data["cartesianCoordinate"]
        rail_y = self.rail(data["auxJoint"][0])
        y_ext = rail_y + self.rail_offset
        return [x,y,z,r,y_ext]
    
    def enable_robot(self):
        print(self.dashboard.GetErrorID())
        self.set_load_params(data = {
            "inertiaX": 0.1,
            "inertiaY": 0.1,
            "inertiaZ": 0,
            "loadValue": 0.1
        })

        #self.enable_rail(True)
        #end effector data
        end_effector = SimpleNamespace(
            load= 0.2,# in kg
            center_mass= SimpleNamespace(
                x= 0.1, # in mm ?
                y= 0.1,
                z= 0.1
            )
        )
        # self.dashboard.AccJ(50) # in %
        # self.dashboard.AccL(50)
        # self.dashboard.SpeedFactor(50)
        time.sleep(1)
        print(f'load={self.get_load_params()}')
        print(f'error={self.dashboard.GetErrorID()}')
        print(f'coord={self.get_arm_and_rail_coordinates()}')
        print(f'user={self.get_user_param()}')
        self.dashboard.SetPayload(end_effector.load,100)
        time.sleep(1)
        print(f'load={self.get_load_params()}')
        print(f'error={self.dashboard.GetErrorID()}')
        print(f'coord={self.get_arm_and_rail_coordinates()}')
        print(f'user={self.get_user_param()}')
        self.set_control_mode(data = {
        "controlMode": "enable"
        })
        time.sleep(1)
        print(f'load={self.get_load_params()}')
        print(f'error={self.dashboard.GetErrorID()}')
        print(f'coord={self.get_arm_and_rail_coordinates()}')
        print(f'user={self.get_user_param()}')
        self.dashboard.EnableRobot(end_effector.load, *end_effector.center_mass.__dict__.values())
        time.sleep(1)
        print(f'load={self.get_load_params()}')
        print(f'error={self.dashboard.GetErrorID()}')
        print(f'coord={self.get_arm_and_rail_coordinates()}')
        print(f'user={self.get_user_param()}')
        # #self.enable_rail(True)
        # self.dashboard.AccJ(50) # in %
        # self.dashboard.AccL(50)
        # self.dashboard.SpeedFactor(50)
        self.dashboard.SetPayload(end_effector.load,100)
        # time.sleep(1)
    def disable_robot(self) :
        self.set_control_mode(data = {
        "controlMode": "disable"
        })
        # #self.enable_rail(False)
        # #time.sleep(1)
        self.dashboard.DisableRobot()

In [ ]:
import math
import json
class Workspace:
    
    def __init__(self,fridge, arm) -> None:
        self.arm = arm
        self.fridge =fridge
        
        self.min_safe_total_y = 433
        self.max_safe_slide = 733+20
        self.slide_fridge = 360
        self.min_safe_slide = 0
        self.min_safe_x = -110
        self.min_safe_z = -120
        self.safe_plate = self.min_safe_x, -220, 0
        self.ot2_shift_slot_y = 132.49 # from CAD OT-2 file
        self.ot2_angular_offset = 1.15 # 1.1
        self.r_0 = -83.5
        
        self.plate_ot2_slot3 = 293, -132.5, -59.5
        self.plate_front_fridge_1_2 = 110, -170, 50,
        self.up_z_offset = 20
        self.cover_z_offset = 7

        self.plate_fridge_slot1 = 277, -80.3, 36.5   #277.184814453125, -80.25714874267578, 36.48689270019531

        self.fridge_angular_offset_1_2 = 1.5 

        self.plate_fridge_slot2 = 274, -228.3, 35.
        
        self.plate_front_fridge_3_4 = 110, -170, 0,
        
        self.fridge_angular_offset_3_4 = 1.5 

        self.plate_fridge_slot3 = 275, -83.0, -43.5

        self.plate_fridge_slot4 = 272.5, -231, -46.5

        self.pos_final_stock_cover_0= [94,
        260.43,
        -171.00,
        99.15,
        703.00]

        self.pos_final_stock_slot_0= [94,
        260.43,
        -163,
        99.15,
        703.00]
        

        self.pos_start_stock_cover= [140.0,
        132.43,
        0,
        53.2,
        703.00]
        
        self.temp_module_z_offset = 80
        self.temp_module_y_offset = -2
        self.temp_module_x_offset = 0
        self.compute_pos()

    def save_data(self, filename):
        self.compute_pos()
        data = { k:v for k,v in vars(self).items() if k not in ['arm','fridge'] }
        with open(filename,'w') as f:
            json.dump(data,f)
    
    def load_data(self, filename) :
        with open(filename,'r') as f:
            data = json.load(f)
        for k,v in data.items():
            self.__dict__[k] = v
        self.compute_pos()

    def compute_pos(self):

        self.r_90 = self.r_0 + 90

        self.pos_front_fridge_1_2 =  [
                self.plate_front_fridge_1_2[0],
                self.plate_front_fridge_1_2[1],
                self.plate_front_fridge_1_2[2],
                self.r_90,
                self.slide_fridge]
        
        self.pos_front_fridge_3_4 =  [
                self.plate_front_fridge_3_4 [0],
                self.plate_front_fridge_3_4 [1],
                self.plate_front_fridge_3_4 [2],
                self.r_90,
                self.slide_fridge]
    
        self.r_slot3 = self.ot2_angular_offset + self.r_90

        self.plate_ot2_slot2 = [
            self.plate_ot2_slot3[0] + self.ot2_shift_slot_y*math.sin(math.radians(self.ot2_angular_offset)),
            self.plate_ot2_slot3[1] + self.ot2_shift_slot_y*math.cos(math.radians(self.ot2_angular_offset)), 
            self.plate_ot2_slot3[2] + 1
        ]
        self.plate_ot2_slot1 =  [
            self.plate_ot2_slot3[0] + 2* self.ot2_shift_slot_y*math.sin(math.radians(self.ot2_angular_offset)),
            self.plate_ot2_slot3[1] + 2* self.ot2_shift_slot_y*math.cos(math.radians(self.ot2_angular_offset)), 
            self.plate_ot2_slot3[2] + 1.0
        ]
        

        self.r_fridge_slot1 = self.fridge_angular_offset_1_2 + self.r_90

        def compute_slot(x,y,z,r,slide):
            up = [x,y,z+self.up_z_offset,r,slide]
            final = [x,y,z,r,slide]
            cover = [x,y,z+self.cover_z_offset,r,slide]
            return up,final,cover
        
        self.pos_up_fridge_slot1, self.pos_final_fridge_slot1, self.pos_cover_fridge_slot1 = compute_slot(
            self.plate_fridge_slot1[0],
            self.plate_fridge_slot1[1],
            self.plate_fridge_slot1[2],
            self.r_fridge_slot1,
            self.pos_front_fridge_1_2[4]
        )

        self.r_fridge_slot2 = self.fridge_angular_offset_1_2 + self.r_90 
        
        self.pos_up_fridge_slot2, self.pos_final_fridge_slot2, self.pos_cover_fridge_slot2 = compute_slot(
            self.plate_fridge_slot2[0],
            self.plate_fridge_slot2[1],
            self.plate_fridge_slot2[2],
            self.r_fridge_slot2,
            self.pos_front_fridge_1_2[4]
        )

        self.r_fridge_slot3 = self.fridge_angular_offset_3_4 + self.r_90 

        self.pos_up_fridge_slot3, self.pos_final_fridge_slot3, self.pos_cover_fridge_slot3 = compute_slot(
            self.plate_fridge_slot3[0],
            self.plate_fridge_slot3[1],
            self.plate_fridge_slot3[2],
            self.r_fridge_slot3,
            self.pos_front_fridge_3_4[4]
        )
        
        self.r_fridge_slot4 = self.fridge_angular_offset_3_4 + self.r_90  

        self.pos_up_fridge_slot4, self.pos_final_fridge_slot4, self.pos_cover_fridge_slot4 = compute_slot(
            self.plate_fridge_slot4[0],
            self.plate_fridge_slot4[1],
            self.plate_fridge_slot4[2],
            self.r_fridge_slot4,
            self.pos_front_fridge_3_4[4]
        )
                

        self.pos_security = [ 
            self.safe_plate[0],
            self.safe_plate[1],
            self.safe_plate[2],
            self.r_0,
            self.max_safe_slide]

        self.pos_rail_origin = self.pos_security[0:-1] +[0]


        self.pos_start_OT2_slot3 = [
            140.0,
            self.plate_ot2_slot3[1],
                0,
            self.r_slot3,
            self.max_safe_slide]

        self.pos_start_OT2_slot2 = [
            220,
        self.plate_ot2_slot2[1],
                0,
            self.r_slot3,
            self.max_safe_slide]

        self.pos_start_OT2_slot1 = [
            140.0,
            self.plate_ot2_slot1[1],
                0,
            self.r_slot3,
            self.max_safe_slide]
        
        self.pos_start_OT2_temp_module_slot3 = [
            self.pos_start_OT2_slot3[0],
            self.pos_start_OT2_slot3[1],
            self.pos_start_OT2_slot3[2] + 25,
            self.pos_start_OT2_slot3[3],
            self.pos_start_OT2_slot3[4]]
        
        self.pos_up_OT2_slot1, self.pos_final_OT2_slot1, self.pos_cover_OT2_slot1 = compute_slot(
            self.plate_ot2_slot1[0],
            self.plate_ot2_slot1[1],
            self.plate_ot2_slot1[2],
            self.r_slot3,
            self.max_safe_slide
        )
        
        self.pos_up_OT2_temp_module_slot3, self.pos_final_OT2_temp_module_slot3, self.pos_cover_OT2_temp_module_slot3 = compute_slot(
            self.plate_ot2_slot3[0] + self.temp_module_x_offset,
            self.plate_ot2_slot3[1] + self.temp_module_y_offset,
            self.plate_ot2_slot3[2] + self.temp_module_z_offset,
            self.r_slot3,
            self.max_safe_slide
        )
        
        self.pos_up_OT2_slot2, self.pos_final_OT2_slot2, self.pos_cover_OT2_slot2 = compute_slot(
            self.plate_ot2_slot2[0],
            self.plate_ot2_slot2[1],
            self.plate_ot2_slot2[2],
            self.r_slot3,
            self.max_safe_slide
        )

        self.pos_up_OT2_slot3, self.pos_final_OT2_slot3, self.pos_cover_OT2_slot3 = compute_slot(
            self.plate_ot2_slot3[0],
            self.plate_ot2_slot3[1],
            self.plate_ot2_slot3[2],
            self.r_slot3,
            self.max_safe_slide
        )
        
        self.pos_up_stock_slot_0 = list(self.pos_final_stock_slot_0)
        self.pos_up_stock_slot_0[2] = self.pos_final_stock_slot_0[2]+20
        
    def calibrate(self,slot):
        x,y,z,r,slide = self.arm.get_arm_and_rail_coordinates()
        if slot == "final_stock_cover_0":
            self.pos_final_stock_cover_0 = [x,y,z,r,slide]
            self.compute_pos()
            return
        if slot == "start_stock_cover":
            self.pos_start_stock_cover = [x,y,z,r,slide]
            self.compute_pos()
            return
        if slot == "final_stock_slot_0":
            self.pos_final_stock_slot_0 = [x,y,z,r,slide]
            self.compute_pos()
            return
        if slot == "security":
            self.safe_plate = x,y,z
            self.r_0 = r
            self.max_safe_slide = slide
            self.compute_pos()
            return
        if slot == "final_OT2_slot3":
            self.plate_ot2_slot3 = x,y,z
            self.max_safe_slide = slide
            self.ot2_angular_offset = r - self.r_90
            self.compute_pos()
            return
        if slot == "final_OT2_slot1":
            self.ot2_angular_offset =math.degrees(math.atan2(y - self.plate_ot2_slot3[1],x - self.plate_ot2_slot3[0]))
            self.compute_pos()
            return
        if slot == "final_fridge_slot1":
            self.plate_fridge_slot1 = x,y,z
            self.fridge_angular_offset_1_2 = r - self.r_90
            self.compute_pos()
            return
        if slot == "final_fridge_slot2":
            self.plate_fridge_slot2 = x,y,z
            self.r_fridge_slot2 = r
            self.compute_pos()
            return
        if slot == "final_fridge_slot3":
            self.plate_fridge_slot3 = x,y,z
            self.fridge_angular_offset_3_4 = r - self.r_90
            self.r_fridge_slot3 = r
            self.compute_pos()
            return
        if slot == "final_fridge_slot4":
            self.plate_fridge_slot4 = x,y,z
            self.r_fridge_slot4 = r
            self.compute_pos()
            return
        print("unknown position name")       

    
    def safe(self, action):
        if action == self.fridge.open_door or action == self.fridge.close_door :
            x,y,z,r,y_slide = self.arm.get_arm_and_rail_coordinates()
            if y + y_slide > self.min_safe_total_y:
                action()
            else:
                print("robot not safe for Fridge action ")
        else:
            print("unknown action")
    
    def do_action_slot(self, slot="slot1", griper_action=None):
        
        def final_move(pos_start,pos_up,pos_final,griper_action=None):
            self.arm.mov_arm_coords_and_rail(*pos_start)
            self.arm.mov_arm_coords_and_rail(*pos_up)
            self.arm.mov_arm_coords_and_rail(*pos_final)
            if griper_action == "break":
                return
            if griper_action :
                griper_action()
            self.arm.mov_arm_coords_and_rail(*pos_up)
            self.arm.mov_arm_coords_and_rail(*pos_start)
        
        def go_back(pos_start,pos_up,pos_final):
            if self.arm.is_close_to(*pos_final):
                self.arm.mov_arm_coords_and_rail(*pos_up)
                self.arm.mov_arm_coords_and_rail(*pos_start)

        def go_back_slot_and_cover(pos_slot_start, pos_up, pos_final_cover, pos_final_slot):
            for pos_final in [pos_final_cover, pos_final_slot]:
                go_back(pos_slot_start,
                        pos_up,
                        pos_final)
        
        if slot == "security" :
            if griper_action :
                griper_action()
            go_back_slot_and_cover(
                self.pos_start_stock_cover,
                self.pos_up_stock_slot_0,
                self.pos_final_stock_cover_0,
                self.pos_final_stock_slot_0
            )
            
            go_back_slot_and_cover(
                self.pos_start_OT2_slot1,
                self.pos_up_OT2_slot1,
                self.pos_cover_OT2_slot1,
                self.pos_final_OT2_slot1
            )
            go_back_slot_and_cover(
                self.pos_start_OT2_slot2,
                self.pos_up_OT2_slot2,
                self.pos_cover_OT2_slot2,
                self.pos_final_OT2_slot2
            )
            go_back_slot_and_cover(
                self.pos_start_OT2_slot3,
                self.pos_up_OT2_slot3,
                self.pos_cover_OT2_slot3,
                self.pos_final_OT2_slot3
            )
            go_back_slot_and_cover(
                    self.pos_start_OT2_temp_module_slot3,
                    self.pos_up_OT2_temp_module_slot3,
                    self.pos_cover_OT2_temp_module_slot3,
                    self.pos_final_OT2_temp_module_slot3
            )
            go_back_slot_and_cover(
                    self.pos_front_fridge_1_2,
                    self.pos_up_fridge_slot1,
                    self.pos_cover_fridge_slot1,
                    self.pos_final_fridge_slot1
            )
            go_back_slot_and_cover(
                    self.pos_front_fridge_1_2,
                    self.pos_up_fridge_slot2,
                    self.pos_cover_fridge_slot2,
                    self.pos_final_fridge_slot2
            )
            go_back_slot_and_cover(
                    self.pos_front_fridge_3_4,
                    self.pos_up_fridge_slot3,
                    self.pos_cover_fridge_slot3,
                    self.pos_final_fridge_slot3
            )
            go_back_slot_and_cover(
                    self.pos_front_fridge_3_4,
                    self.pos_up_fridge_slot4,
                    self.pos_cover_fridge_slot4,
                    self.pos_final_fridge_slot4
            )
            
            if self.arm.is_close_to(*self.pos_start_OT2_temp_module_slot3) :
                self.arm.mov_arm_coords_and_rail(*self.pos_start_OT2_slot3)
            
            if self.arm.is_close_to(*self.pos_start_stock_cover) :
                self.arm.mov_arm_coords_and_rail(*self.pos_start_OT2_slot1)
            if self.arm.is_close_to(*self.pos_start_OT2_slot1) :
                self.arm.mov_arm_coords_and_rail(*self.pos_start_OT2_slot3)
            if self.arm.is_close_to(*self.pos_start_OT2_slot2) :
                self.arm.mov_arm_coords_and_rail(*self.pos_start_OT2_slot3)
            if self.arm.is_close_to(*self.pos_start_OT2_slot3) :
                self.arm.mov_arm_coords_and_rail(*self.pos_security)
            if self.arm.is_close_to(*self.pos_front_fridge_1_2) :
                self.arm.mov_arm_coords_and_rail(*self.pos_security)
                self.safe(self.fridge.close_door)
            if self.arm.is_close_to(*self.pos_front_fridge_3_4) :
                self.arm.mov_arm_coords_and_rail(*self.pos_security)
                self.safe(self.fridge.close_door)
        
        elif slot == "stock_slot_0":
            pos_final = self.pos_final_stock_slot_0
            if self.arm.is_close_to(*self.pos_security):
                self.arm.mov_arm_coords_and_rail(*self.pos_start_OT2_slot3)
            if  ( self.arm.is_close_to(*self.pos_start_OT2_slot2) or 
                    self.arm.is_close_to(*self.pos_start_OT2_slot1) or 
                    self.arm.is_close_to(*self.pos_start_OT2_slot3) ):
                self.arm.mov_arm_coords_and_rail(*self.pos_start_OT2_slot1)
            if self.arm.is_close_to(*self.pos_start_OT2_slot1) :
                self.arm.mov_arm_coords_and_rail(*self.pos_start_stock_cover)
            if self.arm.is_close_to(*self.pos_start_stock_cover):
                final_move(
                    self.pos_start_stock_cover,
                    self.pos_up_stock_slot_0,
                    pos_final,
                    griper_action
                )
                if griper_action == "break":
                    return
                self.arm.mov_arm_coords_and_rail(*self.pos_start_stock_cover)
                self.arm.mov_arm_coords_and_rail(*self.pos_start_OT2_slot1)
                self.arm.mov_arm_coords_and_rail(*self.pos_start_OT2_slot3)
                self.arm.mov_arm_coords_and_rail(*self.pos_security)

        elif slot == "slot1" or slot == "cover_slot1":
            if slot == "slot1":
                pos_final = self.pos_final_OT2_slot1
            else:
                pos_final = self.pos_cover_OT2_slot1
            if self.arm.is_close_to(*self.pos_security):
                self.arm.mov_arm_coords_and_rail(*self.pos_start_OT2_slot3)
            if  ( self.arm.is_close_to(*self.pos_start_OT2_slot2) or 
                    self.arm.is_close_to(*self.pos_start_OT2_slot1) or 
                    self.arm.is_close_to(*self.pos_start_OT2_slot3) ):
                final_move(
                    self.pos_start_OT2_slot1,
                    self.pos_up_OT2_slot1,
                    pos_final,
                    griper_action
                )
                if griper_action == "break":
                    return
                self.arm.mov_arm_coords_and_rail(*self.pos_start_OT2_slot3)
                self.arm.mov_arm_coords_and_rail(*self.pos_security)
        elif slot == "slot2" or slot == "cover_slot2":
            if slot == "slot2":
                pos_final = self.pos_final_OT2_slot2
            else:
                pos_final = self.pos_cover_OT2_slot2
            if self.arm.is_close_to(*self.pos_security):
                self.arm.mov_arm_coords_and_rail(*self.pos_start_OT2_slot3)
            if ( self.arm.is_close_to(*self.pos_start_OT2_slot3) or 
            self.arm.is_close_to(*self.pos_start_OT2_slot1) or 
            self.arm.is_close_to(*self.pos_start_OT2_slot2) ) :
                final_move(
                    self.pos_start_OT2_slot2,
                    self.pos_up_OT2_slot2,
                    pos_final,
                    griper_action
                    )
                if griper_action == "break":
                    return
                self.arm.mov_arm_coords_and_rail(*self.pos_start_OT2_slot3)
            self.arm.mov_arm_coords_and_rail(*self.pos_security)
        elif slot == "slot3" or slot == "cover_slot3":
            if slot == "slot3":
                pos_final = self.pos_final_OT2_slot3
            else:
                pos_final = self.pos_cover_OT2_slot3
            if ( self.arm.is_close_to(*self.pos_security) or 
            self.arm.is_close_to(*self.pos_start_OT2_slot2) or
            self.arm.is_close_to(*self.pos_start_OT2_slot3) ):
                final_move(
                    self.pos_start_OT2_slot3,
                    self.pos_up_OT2_slot3,
                    pos_final,
                    griper_action
                    )
                if griper_action == "break":
                    return
            self.arm.mov_arm_coords_and_rail(*self.pos_security)
        elif slot == "temp_module_slot3" or slot == "cover_temp_module_slot3":
            if slot == "temp_module_slot3":
                pos_final = self.pos_final_OT2_temp_module_slot3
            else:
                pos_final = self.pos_cover_OT2_temp_module_slot3
            if self.arm.is_close_to(*self.pos_security):
                self.arm.mov_arm_coords_and_rail(*self.pos_start_OT2_slot3)
            if  ( self.arm.is_close_to(*self.pos_start_OT2_slot2) or 
                    self.arm.is_close_to(*self.pos_start_OT2_slot1) or 
                    self.arm.is_close_to(*self.pos_start_OT2_temp_module_slot3) or 
                    self.arm.is_close_to(*self.pos_start_OT2_slot3) ):
                final_move(
                    self.pos_start_OT2_temp_module_slot3,
                    self.pos_up_OT2_temp_module_slot3,
                    pos_final,
                    griper_action
                )
                if griper_action == "break":
                    return
                self.arm.mov_arm_coords_and_rail(*self.pos_start_OT2_slot3)
                self.arm.mov_arm_coords_and_rail(*self.pos_security)
        elif slot == "fridge_slot1" or slot == "cover_fridge_slot1":
            if slot == "fridge_slot1":
                pos_final = self.pos_final_fridge_slot1
            else:
                pos_final = self.pos_cover_fridge_slot1
            if self.arm.is_close_to(*self.pos_security) :
                if self.fridge.is_door_close():
                    self.safe(self.fridge.open_door)
            if self.fridge.is_door_open():
                final_move(
                    self.pos_front_fridge_1_2,
                    self.pos_up_fridge_slot1,
                    pos_final,
                    griper_action
                    )
                if griper_action == "break":
                    return
            self.arm.mov_arm_coords_and_rail(*self.pos_security)
            self.safe(self.fridge.close_door)
        elif slot == "fridge_slot2" or slot == "cover_fridge_slot2":
            if slot == "fridge_slot2":
                pos_final = self.pos_final_fridge_slot2
            else:
                pos_final = self.pos_cover_fridge_slot2
            if self.arm.is_close_to(*self.pos_security) :
                if self.fridge.is_door_close():
                    self.safe(self.fridge.open_door)
            if self.fridge.is_door_open():
                final_move(
                    self.pos_front_fridge_1_2,
                    self.pos_up_fridge_slot2,
                    pos_final,
                    griper_action
                )
                if griper_action == "break":
                    return
            self.arm.mov_arm_coords_and_rail(*self.pos_security)
            self.safe(self.fridge.close_door)
        elif slot == "fridge_slot3" or slot == "cover_fridge_slot3":
            if slot == "fridge_slot3":
                pos_final = self.pos_final_fridge_slot3
            else:
                pos_final = self.pos_cover_fridge_slot3
            if self.arm.is_close_to(*self.pos_security) :
                if self.fridge.is_door_close():
                    self.safe(self.fridge.open_door)
            if self.fridge.is_door_open():
                final_move(
                    self.pos_front_fridge_3_4,
                    self.pos_up_fridge_slot3,
                    pos_final,
                    griper_action
                    )
                if griper_action == "break":
                    return
            self.arm.mov_arm_coords_and_rail(*self.pos_security)
            self.safe(self.fridge.close_door)
        elif slot == "fridge_slot4" or slot == "cover_fridge_slot4":
            if slot == "fridge_slot4":
                pos_final = self.pos_final_fridge_slot4
            else:
                pos_final = self.pos_cover_fridge_slot4
            if self.arm.is_close_to(*self.pos_security) :
                if self.fridge.is_door_close():
                    self.safe(self.fridge.open_door)
            if self.fridge.is_door_open():
                final_move(
                    self.pos_front_fridge_3_4,
                    self.pos_up_fridge_slot4,
                    pos_final,
                    griper_action
                )
                if griper_action == "break":
                    return
            self.arm.mov_arm_coords_and_rail(*self.pos_security)
            self.safe(self.fridge.close_door)
        else :
            print(f"slot {slot} is not know")
            pass    

In [ ]:
fridge = Fridge(serial_path="/dev/ttyACM0")

In [ ]:
arm = Dobot()

In [ ]:
ws = Workspace(fridge, arm)

In [ ]:
ws.load_data('20250723_calibration.json')

In [ ]:
arm.connect()
arm.get_robot_status()

In [ ]:
arm.get_load_params()

In [ ]:
arm.get_arm_and_rail_coordinates()

In [ ]:
ws.safe(fridge.close_door)
# To avoid a connexion error with the USB connexion, please start OT-2 modules AFTER loading the DOBOT and fridge code

In [ ]:
arm.enable_robot()

In [ ]:
arm.get_robot_status()

In [ ]:
#slow down movements
arm.set_global_speed_ratio(100)
time.sleep(2)
arm.set_global_speed_ratio(22)

In [ ]:
arm.mov_arm_coords_and_rail(*ws.pos_security) # move to security position

## **DOBOT ERROR DEBUGGING**

In [ ]:
arm.dashboard.GetErrorID()

In [ ]:
arm.print_error_message()

In [ ]:
arm.dashboard.ClearError()

In [ ]:
arm.disable_robot()

In [ ]:
arm.dashboard.ResetRobot()

## **IMPORT OPENTRONS OT-2 API AND INITIALIZE PROTOCOL**

In [2]:
#|HELPER FUNCTIONS|#

# Function to tracking well volumes
def update_volume(plate, well, volume):  
    if plate not in total_volumes: # Labware validation
        raise KeyError(f"Labware {plate} doesn't exist.")
    well = well.upper() # Converts well names to uppercase, avoiding case errors (e.g. a1 and A1 will be identical).  
    if well not in total_volumes[plate]: # Volume initialization for new wells
        total_volumes[plate][well] = 0
    total_volumes[plate][well] += volume # Volume update
    if total_volumes[plate][well] > max_volumes[plate]: # Checking maximum capacity
        raise ValueError(f"The total volume in {plate} well {well} exceeds {max_volumes[plate]} µL.")

# Function to display final well volumes
def display_final_volumes():
    """Displays the volume contained in each well for all labwares."""
    print("\n\nSummary of final well volumes:")
    for plate_name, wells in total_volumes.items():
        print(f"\n\tLabware : {plate_name}")
        if not wells:
            print("\tNo recorded volumes.")
        else:
            for well, volume in wells.items():
                print(f"\tPuit {well} : {volume:.2f} µL")

# Operating time display function
import time
def estimate_pipetting_time(volume, pipette):
    """ Calculates aspirating and dispensing time as a function of flow rates."""
    aspirate_time = volume / pipette.flow_rate.aspirate
    dispense_time = volume / pipette.flow_rate.dispense
    return aspirate_time, dispense_time

def format_time(seconds):
    minutes = int(seconds // 60)
    sec = seconds % 60
    return f"{minutes} min {sec:.2f} sec" if minutes > 0 else f"{sec:.2f} sec"

# Pipette tips counter to pause the protocol every N uses for an OT-2 calibration check
TIP_THRESHOLD = 150  # Global tip threshold (total tips across all pipettes)

# Initialize tip counters as an empty dictionary
tips_used = {} # Pipette names will be added dynamically when used

def _increment_tip_count_by_name(pipette_name):
    """Increment the counter for a single pipette by name."""
    tips_used[pipette_name] = tips_used.get(pipette_name, 0) + 1

def increment_tip_count(ctx, pipette):
    """
    1) Increment the tip counter for this pipette
    2) Check the **total** tip count and pause if the threshold is reached.
    """
    _increment_tip_count_by_name(pipette.name)
    check_total_tip_threshold(ctx)

def check_total_tip_threshold(ctx):
    """
    If the total tip count across all pipettes reaches TIP_THRESHOLD,
    post a comment prompting for a calibration check (no pause),
    and also print the same message to the notebook output.
    """
    total = sum(tips_used.values())
    if total >= TIP_THRESHOLD:
        msg = (
            f"⚠️ Total tip usage across all pipettes has reached {total} tips "
            f"(threshold = {TIP_THRESHOLD}).\n"
            "Please perform a calibration check before continuing."
        )
        ctx.comment(msg)  # Send message to the Opentrons interface
        print(msg)        # Print message to the notebook output

def display_tip_usage():
    """Prints the number of tips used per pipette at the end of the run."""
    print("\nSummary of pipette tips used:\n")
    for name, count in tips_used.items():
        print(f"\tTips used by {name} : {count}") 

def reset_tip_counters():
    """Reset the tip counters for all pipettes."""
    tips_used.clear()
    
# Function for displaying values with only 1 decimal place (rounded up)
def round_up_one_decimal(val):
    return math.ceil(val * 10) / 10

# Function for selecting the appropriate pipette
def choose_pipette_and_speed(volume, viscous=False):
    """
    Chooses the pipette (p20 or p300), pipetting speed, and push_out volume depending on:
      - volume (1–20 µL → p20, >20 µL → p300)
      - viscosity (viscous=False → non-viscous, viscous=True → viscous)
    Returns (pipette, speed, push_out).
    """
    if 1 <= volume <= 20:
        pip = p20
        speed = p20_pipetting_speed_viscous if viscous else p20_pipetting_speed_non_viscous
        push_out = p20_push_out
    elif volume > 20:
        pip = p300
        speed = p300_pipetting_speed_viscous if viscous else p300_pipetting_speed_non_viscous
        push_out = p300_push_out
    else:
        raise ValueError(f"❌ Volume {volume} µL out of supported range (≥1 µL).")
    return pip, speed, push_out

print("✅ Helper functions successfully initializeds!")

✅ Helper functions successfully initializeds!


In [ ]:
import os
os.system("systemctl stop opentrons-robot-server")
print("✅ Opentrons robot server stopped!")

In [ ]:
#|PROTOCOL EXECUTION|#
start_time = time.time()  # start of the timer
import opentrons.execute
from opentrons import protocol_api, types
import os
import math
import json  # JavaScript Object Notation
import random # import the random functions
from IPython.display import HTML, display  # Allow to display HTML image in Python code
import csv  # To load csv files

# Launch API V2.19
ctx = opentrons.execute.get_protocol_api('2.19')

end_time = time.time() # end of the timer
elapsed_time = end_time - start_time  # Calculation of elapsed time
print(f"✅ API successfully initializeds in {format_time(elapsed_time)}!")  # 2 min

ctx.set_rail_lights(True)  # Turn on ligths

In [ ]:
#|PROTOCOL SIMULATION|#
import opentrons.simulate
from opentrons import protocol_api, types
import math
import json  # JavaScript Object Notation
import random # import the random functions
from IPython.display import HTML, display  # Allow to display HTML image in Python code
import csv  # To load csv files

# Launch API SIMULATOR V2.19
ctx = opentrons.simulate.get_protocol_api('2.19')
print("✅ API SIMULATOR successfully initializeds!")

## **LOAD CSV FILES**

In [ ]:
# Load CSV
step1p1_gga_master_mix_buffer = 'CSV/GGA/UDHR_construction/20250506_step1p1_gga_master_mix_buffer.csv'
step1p2_gga_master_mix_enzyme = 'CSV/GGA/UDHR_construction/20250506_step1p2_gga_master_mix_enzyme.csv'
step1p3_gga_master_mix_mixing = 'CSV/GGA/UDHR_construction/20250506_step1p3_gga_master_mix_mixing.csv'
step2_gga_master_mix_mixing = 'CSV/GGA/UDHR_construction/20250506_step2_gga_master_mix_splitting.csv'
step4p1_adding_eblocks = 'CSV/GGA/UDHR_construction/20250506_step4p1_add_eblocks.csv'
step4p2_mixing = 'CSV/GGA/UDHR_construction/20250506_step4p2_mixing.csv'
step6p1_gga_transfer_to_tc = 'CSV/GGA/UDHR_construction/20250506_step6p1_gga_transfer_to_tc.csv'
step7p1_gga_transfer_from_tc = 'CSV/GGA/UDHR_construction/20250506_step7_gga_transfer_from_tc .csv'

# List of CSV files whose loading must be verified
csv_files = [
    step1p1_gga_master_mix_buffer,
    step1p2_gga_master_mix_enzyme,
    step1p3_gga_master_mix_mixing,
    step2_gga_master_mix_mixing,
    step4p1_adding_eblocks,
    step4p2_mixing,
    step6p1_gga_transfer_to_tc,
    step7p1_gga_transfer_from_tc
]

required_columns = {'source', 'destination', 'source_labware', 'destination_labware'}

for csv_file in csv_files:
    try:
        with open(csv_file, 'r', newline='') as file:
            reader = csv.DictReader(file)
            
            if reader.fieldnames is None:
                raise ValueError(f"Error: The file '{csv_file}' is empty or improperly formatted.")

            missing_columns = required_columns - set(reader.fieldnames)
            if missing_columns:
                raise ValueError(f"Error: The file '{csv_file}' is missing columns: {missing_columns}")

        print(f"✅ CSV file '{csv_file}' is correctly formatted.")

    except FileNotFoundError:
        print(f"❌ Error: The file '{csv_file}' cannot be found.")
    except ValueError as e:
        print(e)
    except Exception as e:
        print(f"⚠ Unexpected error with file '{csv_file}': {e}")

# Function to display csv source and destination|#
def print_wells_from_csv(csv_path: str):
    """
    Reads the CSV file and displays for each line:
      Source: <source_well> (labware: <source_labware>),
      Destination: <destination_well> (labware: <destination_labware>)
    """
    try:
        with open(csv_path, mode='r', newline='') as f:
            reader = csv.DictReader(f)
            #  Quick check of expected columns
            cols = set(reader.fieldnames or [])
            required = {'source', 'destination', 'source_labware', 'destination_labware'}
            missing = required - cols
            if missing:
                print(f"❌ Missing columns in CSV.: {', '.join(missing)}")
                return

            print("Transfer operations:")
            for i, row in enumerate(reader, start=1):
                src = row['source'].strip()
                dst = row['destination'].strip()
                lw_src = row['source_labware'].strip()
                lw_dst = row['destination_labware'].strip()
                print(
                    f"→ Line {i}\n"
                    f"Src: {src:>3} (lab: {lw_src})\n"
                    f"Dst: {dst:>3} (lab: {lw_dst})\n"
                )

    except FileNotFoundError:
        print(f"❌ File not found: {csv_path}")
    except Exception as e:
        print(f"❌ Error reading CSV: {e}")

## **LOAD AND SET HARWARES AND LABWARES**

In [ ]:
# Create a JSON library that defines the pipette numbers and mounting positions:
def get_values(*names):
    _all_values = json.loads("""{"p300_single_gen2_mount":"left","p20_single_gen2_mount":"right"}""")
    return [_all_values[n] for n in names]

result = get_values("p300_single_gen2_mount", "p20_single_gen2_mount")
p300_single_gen2_mount, p20_single_gen2_mount = result
print(f"p300_single_gen2_mount: {p300_single_gen2_mount}, p20_single_gen2_mount: {p20_single_gen2_mount}")

#|LOADING HARDWARE MODULES|#
tc_mod = ctx.load_module('thermocyclerModuleV2') # USB position 1
print("[LOADING HARDWARE MODULES] Thermocycler Module loaded.")
temp_mod_pos03 = ctx.load_module('temperature module gen2', 3) # USB position 2
print("[LOADING HARDWARE MODULES] Temperature Module Gen2 loaded at position 3.")
temp_mod_pos09 = ctx.load_module('temperature module gen2', 4) # USB position 3
print("[LOADING HARDWARE MODULES] Temperature Module Gen2 loaded at position 4.")

#|LOADING LABWARE ON HARDWARE MODULES|#
tc_plate = tc_mod.load_labware("nest_96_wellplate_100ul_pcr_full_skirt")
print("[LOADING LABWARE] Labware 'nest_96_wellplate_100ul_pcr_full_skirt' loaded on Thermocycler.")
aluminumblock_nest_96_wellplate_100ul_pos03_from_fridge = temp_mod_pos03.load_labware("opentrons_96_aluminumblock_nest_wellplate_100ul")
print("[LOADING LABWARE] Labware 'opentrons_96_aluminumblock_nest_wellplate_100ul' loaded from fridge slot 1 on module at position 3.")
aluminumblock_nest_24_500µL_pos09 = temp_mod_pos09.load_labware("opentrons_24_aluminumblock_nest_0.5ml_screwcap")
print("[LOADING LABWARE] Labware 'opentrons_24_aluminumblock_nest_0.5ml_screwcap' loaded on module at position 9.")
trash = ctx.fixed_trash
print("[LOADING LABWARE Fixed trash loaded.")

# Import custom labware:
    # 24 position aluminium block with 
        # - the top row A1-A6 being for 2mL tubes (snapcap)
        # - the remaining 18 wells (B1 - D6) being for 0.5mL tubes (screwcap)
#with open('opentrons_aluminium_block_2ml_0.5ml_tubes.json') as labware_file:
#    labware_def = json.load(labware_file)
#    aluminumblock_nest_24_500µL_pos09 = temp_mod_pos09.load_labware_from_definition(labware_def)
#    print("[LOADING LABWARE] Labware 'opentrons_aluminium_block_2ml_0.5ml_tubes' loaded on module at position 9.")


# Build a dictionary to directy use csv column in the code
labware_map = {
    'aluminumblock_nest_96_wellplate_100ul_pos03_from_fridge_slot01': aluminumblock_nest_96_wellplate_100ul_pos03_from_fridge,
    'aluminumblock_nest_96_wellplate_100ul_pos03_from_fridge_slot02': aluminumblock_nest_96_wellplate_100ul_pos03_from_fridge,
    'aluminumblock_nest_96_wellplate_100ul_pos03_from_fridge_slot03': aluminumblock_nest_96_wellplate_100ul_pos03_from_fridge,
    'aluminumblock_nest_96_wellplate_100ul_pos03_from_fridge_slot04': aluminumblock_nest_96_wellplate_100ul_pos03_from_fridge,
    'aluminumblock_nest_24_500uL_pos09':                 aluminumblock_nest_24_500µL_pos09,
    'tc_plate':                                         tc_plate,
}

#|LOADING TIP RACKS|#
#tipracks_p300_slot06 = ctx.load_labware('opentrons_96_filtertiprack_200ul', 6)
#print("[LOADING TIP RACKS] P300 tip rack loaded in slot 9.")
tipracks_p20_slot05 = ctx.load_labware('opentrons_96_filtertiprack_20ul', 5)
print("[LOADING TIP RACKS] P20 tip rack loaded in slot 5.")
tipracks_p20_slot02 = ctx.load_labware('opentrons_96_filtertiprack_20ul', 2)
print("[LOADING TIP RACKS] P20 tip rack loaded in slot 2.")
tipracks_p300_slot01 = ctx.load_labware('opentrons_96_filtertiprack_200ul', 1)
print("[LOADING TIP RACKS] P300 tip rack loaded in slot 1.")


#|LOADING PIPETTES|#
p300 = ctx.load_instrument('p300_single_gen2', p300_single_gen2_mount, tip_racks=[tipracks_p300_slot01])
print(f"[LOADING PIPETTES] P300 pipette loaded on mount {p300_single_gen2_mount}.")

p20 = ctx.load_instrument('p20_single_gen2', p20_single_gen2_mount, tip_racks=[tipracks_p20_slot05, tipracks_p20_slot02])
print(f"[LOADING PIPETTES] P20 pipette loaded on mount {p20_single_gen2_mount}.")

print("✅ Harwares and labwares are set and load!")

## **HARWARES CONFIGURATION**

### ⚠ | IMPORTANT ADVICES FOR HARDWARES CONFIGURATION | ⚠
To have a correct temperature regulation of the liquid in the 'opentrons_24_aluminumblock_nest_0.5ml_screwcap' you have to put 1 mL of cold water inside the hole before to put the NEST 0.5 mL on it. You also have to monitoring the temperature of the liquid insied the aluminium block by using a temperature sensor emerging in water. Launch the protocol only when this sensor indicate the expected temperature. With this procedure you don't have many evaporation --> avoid to have "RAPID Slit Seal" on microtubes.

To reduce the evaporation on 'opentrons_96_aluminumblock_nest_wellplate_100ul' you must use the "RAPID Slit Seal" from BioChromato and add 100 µL of water in empty positions!

To avoid / reducing the risk of having an "USB error" with the themocycler (which avoid to use it), you have to first launch the Opentrons Software (after started OT-2) and use it for do something with the themocycler, like moving the lid.

In [ ]:
#|HARDWARE CONFIGURATIONS|#
tc_mod.open_lid()
print("✅ Thermocycler lid opened.")

In [ ]:
#|HARDWARE CONFIGURATIONS|#
tc_mod.close_lid()
print("✅ Thermocycler lid closed.")

In [ ]:
#|HARDWARE CONFIGURATIONS|#
tc_mod.set_block_temperature(temperature=4)
print("✅ Temperature set to 4°C on thermocycler block.")
# During API execution, we need to wait for the tempertaure module to reach 4°C before continuing to run the code.

In [ ]:
# Shutdown temperature block of the thermocycler
tc_mod.deactivate_block()

In [ ]:
#|HARDWARE CONFIGURATIONS|#
temp_mod_pos03.set_temperature(4)
print("✅ Temperature set to 4°C on module at position 3.")
# During API execution, we need to wait for the tempertaure module to reach 4°C before continuing to run the code.

In [ ]:
#|HARDWARE CONFIGURATIONS|#
temp_mod_pos09.set_temperature(4)
print("✅ Temperature set to 4°C on module at position 9.")
# During API execution, we need to wait for the tempertaure module to reach 4°C before continuing to run the code.

## **OFFSETS CONFIGURATION**

### ⚠ | IMPORTANT ADVICES FOR OFFSETS CONFIGURATION | ⚠
1) Do the calibration check on the OT-2 software to have offset values.

2) Report the offsets.

3) Apply the p300 correction offset for 'aluminumblock_nest_24_500µL_pos09': don't know why, but we have an offsets issue with this pipette when working on this labware. --> I HAVE THE IMPRESSION THAT WE DON'T NEED TO DO THIS ANYMORE SINCE WE SHUTDOWN THE OT-2 SERVOR!

4) Manually check and CORRECT offsets 

In [ ]:
#|LABWARE OFFSETS|#
# /!\ Do calibration without slit seal film on the 96 wp

tipracks_p300_slot01.set_offset(x=0.7, y=0.2, z=0.2)

tipracks_p20_slot02.set_offset(x=0.7, y=-0.7, z=-0.4)

tipracks_p20_slot05.set_offset(x=0.5, y=0.2, z=-0.5)

# /!\ Close TC lid once before do the calibration because it can affect z axis if 96 wp isn't well inserted
tc_plate.set_offset(x=0.2, y=-1.0, z=0.1)

# /!\ Apply correction on z axis: error on OT-2 calibration check software
aluminumblock_nest_24_500µL_pos09.set_offset(x=0.5, y=0.6, z=-1.0) 

aluminumblock_nest_96_wellplate_100ul_pos03_from_fridge.set_offset(x=0.2, y=0.8, z=0.0)

print("✅ Offsets set for all labwares.")

In [ ]:
# !! STOP !! --> I HAVE THE IMPRESSION THAT WE DON'T NEED TO DO THIS ANYMORE SINCE WE SHUTDOWN THE OT-2 SERVOR!

# TO BE EXECUTED BEFORE ANY MOVE_TO CALL IF EXECUTED AFTER THE P300 NO LONGER WORKS

# === 1) New offsets (mm) ===
x_offset = 0.0
y_offset = -0.8
z_offset = 0.0

# === 2) Capture original method ===
original_move_to = p300.move_to

# === 3) Imports ===
from opentrons.types import Point, Location

# === 4) Wrapper target ===
def move_to_with_xyz_offset(loc, *args, **kwargs):
    # On ne touche qu'aux déplacements vers les labwares spécifiques
    if hasattr(loc, 'point') and hasattr(loc, 'labware') \
       and loc.labware in [aluminumblock_nest_24_500µL_pos09, tc_plate]:
        # Recalcule du point avec offset
        new_pt = Point(
            loc.point.x + x_offset,
            loc.point.y + y_offset,
            loc.point.z + z_offset
        )
        loc = Location(new_pt, loc.labware)
    # Sinon on laisse loc tel quel
    return original_move_to(loc, *args, **kwargs)

# === 5) Patch p300 ===
p300.move_to = move_to_with_xyz_offset

print("✅ p300 Offsets set for aluminumblock_nest_24_500µL_pos09 and tc_plate.")

### Start of manual calibration check

In [ ]:
# Setting up the manual calibration check
ctx.home()
p300.default_speed = 50
p20.default_speed = 50
p300.pick_up_tip()
p20.pick_up_tip()

### Check deck position 09

In [ ]:
# p300 check X and Y axis on deck position 09
p300.move_to(aluminumblock_nest_24_500µL_pos09['A1'].top(z=0))

In [ ]:
# p300 check Z axis on deck position 09
p300.move_to(aluminumblock_nest_24_500µL_pos09['A1'].bottom(z=0.2))

In [ ]:
# p20 check X and Y axis on deck position 09
p20.move_to(aluminumblock_nest_24_500µL_pos09['A1'].top(z=0))

In [ ]:
# p20 check Z axis on deck position 09
p20.move_to(aluminumblock_nest_24_500µL_pos09['A1'].bottom(z=0.2))

### Check deck position 03

In [ ]:
# p300 check X and Y axis on deck position 03
p300.move_to(aluminumblock_nest_96_wellplate_100ul_pos03_from_fridge['A1'].top(z=0))

In [ ]:
# p300 check Z axis on deck position 03
p300.move_to(aluminumblock_nest_96_wellplate_100ul_pos03_from_fridge['A1'].bottom(z=0.2))

In [ ]:
# p20 check X and Y axis on deck position 03
p20.move_to(aluminumblock_nest_96_wellplate_100ul_pos03_from_fridge['A1'].top(z=0))

In [ ]:
# p20 check Z axis on deck position 03
p20.move_to(aluminumblock_nest_96_wellplate_100ul_pos03_from_fridge['A1'].bottom(z=0.2))

### Check thermocycler position

In [ ]:
# p300 check X and Y axis on thermocycler
p300.move_to(tc_plate['A1'].top(z=0))

In [ ]:
# p300 check Z axis on thermocycler
p300.move_to(tc_plate['A1'].bottom(z=0.2))

In [ ]:
# p20 check X and Y axis on thermocycler
p20.move_to(tc_plate['A1'].top(z=0))

In [ ]:
# p20 check Z axis on thermocycler
p20.move_to(tc_plate['A1'].bottom(z=0.2))

### End of manual calibration check

In [ ]:
# End of the manual calibration check
p300.default_speed = 400
p20.default_speed = 400
p300.drop_tip()
p20.drop_tip()
ctx.home()

<hr style="height: 5px; background-color: black; border: none;">

# <div style="text-align: center;"><font style="color:Black ;"> **PART 1 - GOLDEN GATE ASSEMBLY PROTOCOL** </div></font>

<hr style="height: 5px; background-color: black; border: none;">

<hr style="height: 2px; background-color: black; border: none;">

## 🛑 | EXPERIMENT CONFIGURATIONS | 🛑
***

In [3]:
#|OT-2 DECK VIEW|#
display(HTML('<img src="images/OT-2_deck.png" width="400">'))

#|DESCRIPTION OF REAGENTS FOR 1 REACTION|#
gga_final_reaction_volume = 20.0                                  # (µL)
t4_dna_ligase_buffer_10x = gga_final_reaction_volume / 10         # 2.0 µL
nebridge_gga = 2.0                                                # NEBridge® Golden Gate Assembly (µL)
eBlock_raw_idt = 2.0                                              # Approximate normalized concentration IDT 10 ng/µL
number_of_eblocks = 8                                             # per BigBlocks to assembly
all_eblocks_raw_idt = eBlock_raw_idt * number_of_eblocks          # 16.0 µL
qsp_mq_water = gga_final_reaction_volume - (t4_dna_ligase_buffer_10x + all_eblocks_raw_idt + nebridge_gga)  # 0.0 µL

#|PROTOCOL SPECIFICATIONS|#
n_bigblocks = 12
n_replicas_gga = 1
n_replicas_pcr = 1

# Overage factors: should be high before having spliting operation (pipetting errors accumulation)
overage_factor_before_protocol_launch = 1.9                   # Initial volume of reagents before starting the protocol 
overage_factor_during_master_mix_building_step= 1.6           # During GGA Master Mix building: HIGH VALUE BECAUSE SPLITING
overage_factor_during_splitting_in_mini_master_mix_step= 1.2  # During splitting  GGA Master Mix in GGA mini Master Mix
overage_factor_in_tc = 1.0                                    # In the thermocyler for GGA incubation
overage_factor_after_tc_run = 0.90                            # Post-GGA run in thermocycler: small lost due to evaporation and absence of centrifugation

#|REAGENTS VOLUMES REQUIRED|#
volume_t4_dna_ligase_buffer_10x = t4_dna_ligase_buffer_10x * n_bigblocks * n_replicas_gga * overage_factor_before_protocol_launch
volume_nebridge_gga = nebridge_gga * n_bigblocks * n_replicas_gga * overage_factor_before_protocol_launch
volume_eBlock_raw_idt = eBlock_raw_idt * n_replicas_gga * overage_factor_during_master_mix_building_step
volume_qsp_mq_water = qsp_mq_water * n_bigblocks * n_replicas_gga * overage_factor_before_protocol_launch

#|OT-2 SPECIFICATIONS|#
# Pipettes
p20_max_capacity = 20                        # Volume capacity in µL
p300_max_capacity = 200                      # Volume capacity in µL

# Pipettes z-axis adjustments
top_z_slit_seal_microwell = 3                # Presence of Asynt 'Slit Seal': adjust z position (in mm)
top_z_slit_seal_microtube = 7                # Presence of Asynt 'Slit Seal': adjust z position (in mm)
bottom_z_microwell = 0.20                    # Adjust pipette z position on tube bottom (aspirate and dispense position) (in mm)
bottom_z_microtube = 0.20                    # Adjust pipette z position on tube bottom (aspirate and dispense position) (in mm)
bottom_z_with_liquid_microtube = 2.00        # Adjust pipette z position on tube when liquid is already in the tube (for aspiration only) (in mm)
bottom_z_with_liquid_microwell = 1.50        # Adjust pipette z position on tube when liquid is already in the tube (for aspiration only) (in mm)
top_z_blow_out_microwell = -3.0              # Adjust blow out z position (in mm)
top_z_blow_out_microtube = -7.5              # Adjust blow out z position (in mm): 0.1 mm bottom of max volume of the 0.5 mL microtube

# Pipettes speed adjustments
speed_outside_labware = 400                  # No colission risks (in mm/sec)
speed_inside_labware = 10                    # Presence of Asynt 'Slit Seal': adjust pipette speed (in mm/sec)
speed_inside_labware_viscous = 3             # Viscous liquid: adjust pipette speed (in mm/sec)
p300_pipetting_speed_non_viscous = 50.0      # µL/sec
p300_pipetting_speed_viscous = 10.0          # µL/sec
p20_pipetting_speed_non_viscous = 10.0       # µL/sec
p20_pipetting_speed_viscous = 5.0            # µL/sec

p20.flow_rate.blow_out = 0.5                 # µL/sec determine using Opentrons application note for viscous liquid
p300.flow_rate.blow_out = 4                  # µL/sec determine using Opentrons application note for viscous liquid


# Pipettes push-out adjustments    
# /!\ Push out is a cool feature to dispense all the volume, BUT IT'S BUGED: aspire instead of dispense /!\
p20_push_out = 0                             # p20 push out volume, beware of bubbles (in µL)
p300_push_out = 0                            # p300 push out volume, beware of bubbles (in µL)

# Delay post dispense
delay = 2                                    # sec
delay_viscous = 10                           # sec

#|PRINTS|#
print("-------|DURATION|-------")
print("The protocol takes 4 hours & 52 minutes (approximately)")
print("The protocol uses 146 P20 & 3 P300 tips")
print()
print("-------|VOLUMES|-------")

volumes = {
    "T4 DNA ligase buffer 10X": (t4_dna_ligase_buffer_10x, volume_t4_dna_ligase_buffer_10x),
    "NEBridge® Golden Gate Assembly": (nebridge_gga, volume_nebridge_gga),
    "eBlock raw IDT at 10.0 ng/µL": (eBlock_raw_idt, volume_eBlock_raw_idt),
    "mQ water": (qsp_mq_water, volume_qsp_mq_water)
}

for name, (one, total) in volumes.items():
    print(f"{name} (one reaction) = {round_up_one_decimal(one):.1f} µL")
    print(f"TOTAL {name} (all reactions) = {round_up_one_decimal(total):.1f} µL\n")

steps = [
    ("STEP 1.1", step1p1_gga_master_mix_buffer),
    ("STEP 1.2", step1p2_gga_master_mix_enzyme),
    ("STEP 1.3", step1p3_gga_master_mix_mixing),
    ("STEP 2", step2_gga_master_mix_mixing),
    ("STEP 4.1", step4p1_adding_eblocks),
    ("STEP 4.2", step4p2_mixing),
    ("STEP 6.1", step6p1_gga_transfer_to_tc),
    ("STEP 7.1", step7p1_gga_transfer_from_tc)    
]

for step_name, csv_data in steps:
    print(f"-------|{step_name}|-------")
    print_wells_from_csv(csv_data)
    print()

NameError: name 'HTML' is not defined

<hr style="height: 2px; background-color: black; border: none;">

## 🖐 | **STEP 0 - HUMAN** | 🖐
### Settings OT-2 deck <b>BEFORE</b> the execution of Golden Gate Assembly protocol
***

T4 DNA ligase buffer 10X: 0.5 mL tube B2 ; NEBridge® Golden Gate Assembly: 0.5 mL tube B3; mQ water: NONE ; eBlock raw IDT at 10.0 ng/µL: 96-wp plate -- For volume, check above.

<hr style="height: 2px; background-color: black; border: none;">

## **🤖 | STEP 1 - OPENTRONS OT-2 ROBOT** | 🤖
### Prepare the Golden Gate Master mix w/o eBlocks

***

In [ ]:
##|--STEP 1.1 PREPARE GOLDEN-GATE-ASSEMBLY W/O EBLOCKS MASTER MIX (HIGH VOLUME AND LOW VISCOSITY FIRST): add T4 DNA ligase buffer 10X--|##

# Display only the new instructions generated in this block code, without displaying those that have been executed previously
start_index = len(ctx.commands())

# Reset the total volumes and maximum capacities of labwares
total_volumes = {'aluminumblock_nest_96_wellplate_100ul_pos03_from_fridge_slot01': {}, 'tc_plate': {}, 'aluminumblock_nest_24_500uL_pos09': {}}
max_volumes = {'aluminumblock_nest_96_wellplate_100ul_pos03_from_fridge_slot01': 100, 'tc_plate': 100, 'aluminumblock_nest_24_500uL_pos09': 2000}

# Source(s) and destination(s)
print_wells_from_csv(step1p1_gga_master_mix_buffer)

# Define the volume to be transferred according to the protocol conditions
volume = t4_dna_ligase_buffer_10x * n_bigblocks * n_replicas_gga * overage_factor_during_master_mix_building_step
print(f"🔢 Total volume to transfer (T4 DNA ligase buffer 10X) = {volume:.2f} µL")

# Determine if the liquid is viscous
viscous_flag = True
print(f"💧 Is the liquid viscous? {'Yes' if viscous_flag else 'No'}")

# Pipette and corresponding speed selection
pipette, pipetting_speed, push_out = choose_pipette_and_speed(volume, viscous=viscous_flag)
print(f"🧪 Selected pipette: {'p20' if pipette == p20 else 'p300'}")
print(f"🚀 Pipetting speed: {pipetting_speed} µL/sec")

# Maximum capacity depending on selected pipette
pipette_max_volume = 20 if pipette == p20 else 200
print(f"📏 Max pipette capacity: {pipette_max_volume} µL")

# Number of pipetting operations required
number_of_pipetting_operation = math.ceil(volume / pipette_max_volume)
print(f"🔁 Number of pipetting operations: {number_of_pipetting_operation}")

# Volume for each pipetting operation
tip_volume = volume / number_of_pipetting_operation
print(f"📦 Volume per operation: {tip_volume:.2f} µL")

# Apply the speed to the selected pipette
pipette.flow_rate.aspirate = pipetting_speed
pipette.flow_rate.dispense = pipetting_speed
pipette.flow_rate.blow_out = pipetting_speed

# Display applied speeds
print(f"⚙️ Flow rate settings for pipette {'p20' if pipette == p20 else 'p300'}:")
print(f"  • Aspirate speed: {pipette.flow_rate.aspirate} µL/sec")
print(f"  • Dispense speed: {pipette.flow_rate.dispense} µL/sec")
print(f"  • Blow out speed: {pipette.flow_rate.blow_out} µL/sec")

#__________________________________________________________________________________________
#|RUN PROTOCOL|#

start_time = time.time() # Start of the timer
ctx.home()   # Important to keep the calibration between the steps

try:
    with open(step1p1_gga_master_mix_buffer, 'r') as file:
        reader = csv.DictReader(file)
        rows = list(reader)
        if not rows:
            print("❌ Error: CSV file is empty or has no data rows.")
        else:
            print("✅ CSV successfully read!")

            for row in rows:
                src_well       = row['source']
                dst_well       = row['destination']
                lab_src_name   = row['source_labware']
                lab_dst_name   = row['destination_labware']

                # → here we go from the chain to the container
                lab_src = labware_map[lab_src_name]
                lab_dst = labware_map[lab_dst_name]

                # MIXING SOURCE
                pipette.default_speed = speed_outside_labware
                pipette.pick_up_tip()
                pipette.move_to(lab_src[src_well].top(z=top_z_blow_out_microtube))                
                pipette.default_speed = speed_inside_labware
                pipette.move_to(lab_src[src_well].bottom(z=bottom_z_microtube))                        
                # Manual mixing without Opentrons 'mix' command
                for cycle in range(3):
                    pipette.aspirate(tip_volume)
                    pipette.dispense(tip_volume)
                    ctx.delay(seconds=delay_viscous)
                pipette.default_speed = speed_inside_labware_viscous   # Exit slowly from the liquid
                pipette.move_to(lab_src[src_well].top(z=top_z_blow_out_microtube))
                ctx.delay(seconds=delay)   # Before blow out to allow liquid to settle
                # Blow out and clean pipette tip
                pipette.blow_out()
                pipette.default_speed = speed_inside_labware
                pipette.move_to(lab_src[src_well].bottom(z=bottom_z_with_liquid_microtube))   # Like a 'touch tip' inside liquid: avoid to have to much liquid lost
                pipette.default_speed = speed_inside_labware_viscous   # Exit slowly from the liquid
                pipette.move_to(lab_src[src_well].top(z=top_z_blow_out_microtube))                
                # End of step
                pipette.default_speed = speed_outside_labware
                pipette.drop_tip()
                increment_tip_count(ctx, pipette)                   
 
                for _ in range(number_of_pipetting_operation):         

                    # Update total tip_volumes
                    update_volume(lab_src_name, src_well, -tip_volume)  # Subtract from source well
                    update_volume(lab_dst_name, dst_well, tip_volume)   # Add to destination well                 

                    # Change tip between operation
                    pipette.default_speed = speed_outside_labware
                    pipette.pick_up_tip()
                    
                    # ASPIRATE FROM SOURCE
                    pipette.move_to(lab_src[src_well].top(z=top_z_blow_out_microtube))
                    pipette.default_speed = speed_inside_labware
                    pipette.move_to(lab_src[src_well].bottom(z=bottom_z_microtube))
                    pipette.aspirate(tip_volume)
                    ctx.delay(seconds=delay_viscous)
                    pipette.default_speed = speed_inside_labware_viscous   # Exit slowly from the liquid
                    pipette.move_to(lab_src[src_well].top(z=top_z_blow_out_microtube))
                    pipette.default_speed = speed_outside_labware 
            
                    # DISPENSE TO DESTINATION                                
                    # Dispense
                    pipette.move_to(lab_dst[dst_well].top(z=top_z_blow_out_microtube))
                    pipette.default_speed = speed_inside_labware 
                    pipette.move_to(lab_dst[dst_well].bottom(z=bottom_z_microtube))
                    pipette.dispense(tip_volume, push_out=push_out)   # No mixing: this is the first thing added to the tube
                    ctx.delay(seconds=delay_viscous)
                    # Blow out and clean pipette tip
                    pipette.default_speed = speed_inside_labware_viscous   # Exit slowly from the liquid
                    pipette.move_to(lab_dst[dst_well].top(z=top_z_blow_out_microtube))
                    ctx.delay(seconds=delay)   # Before blow out to allow liquid to settle
                    pipette.blow_out()
                    pipette.default_speed = speed_inside_labware                    
                    pipette.move_to(lab_dst[dst_well].bottom(z=bottom_z_with_liquid_microtube))    # Like a 'touch tip' inside liquid: avoid to have to much liquid lost
                    pipette.default_speed = speed_inside_labware_viscous   # Exit slowly from the liquid
                    pipette.move_to(lab_dst[dst_well].top(z=top_z_blow_out_microtube))
                    # End of step
                    pipette.default_speed = speed_outside_labware    
                    pipette.drop_tip()
                    increment_tip_count(ctx, pipette)

except FileNotFoundError:
    print(f"❌ Error: The file '{step1p1_gga_master_mix_buffer}' cannot be found.")
except Exception as e:
    print(f"❌ Unexpected error: {e}")

step1p1_time = time.time() - start_time # Calculation of elapsed time
ctx.comment(f"🤖 Step 1.1 Golden Gate Assembly Master Mix preparation: mQ water addition completed in {step1p1_time:.2f} seconds.")

# Displays
for instruction in ctx.commands()[start_index:]: # Display only the commands executed in this block
    print(instruction)
display_final_volumes() # Display summary of final volumes
display_tip_usage() # Display number of tips used

#🤖 Step 1.1 completed in 3 minutes 19 seconds using 2 P300 tips

In [ ]:
##|--STEP 1.2 PREPARE GOLDEN-GATE-ASSEMBLY W/O EBLOCKS MASTER MIX (HIGH VOLUME AND LOW VISCOSITY FIRST): add NEBridge® Golden Gate Assembly--|##

# Display only the new instructions generated in this block code, without displaying those that have been executed previously
start_index = len(ctx.commands())

# Reset the total volumes and maximum capacities of labwares
total_volumes = {'aluminumblock_nest_96_wellplate_100ul_pos03_from_fridge_slot01': {}, 'tc_plate': {}, 'aluminumblock_nest_24_500uL_pos09': {}}
max_volumes = {'aluminumblock_nest_96_wellplate_100ul_pos03_from_fridge_slot01': 100, 'tc_plate': 100, 'aluminumblock_nest_24_500uL_pos09': 2000}

# Source(s) and destination(s)
print_wells_from_csv(step1p2_gga_master_mix_enzyme)

# Define the volume to be transferred according to the protocol conditions
volume = nebridge_gga * n_bigblocks * n_replicas_gga * overage_factor_during_master_mix_building_step
print(f"🔢 Total volume to transfer (NEBridge® Golden Gate Assembly) = {volume:.2f} µL")

# Determine if the liquid is viscous
viscous_flag = True
print(f"💧 Is the liquid viscous? {'Yes' if viscous_flag else 'No'}")

# Pipette and corresponding speed selection
pipette, pipetting_speed, push_out = choose_pipette_and_speed(volume, viscous=viscous_flag)
print(f"🧪 Selected pipette: {'p20' if pipette == p20 else 'p300'}")
print(f"🚀 Pipetting speed: {pipetting_speed} µL/sec")

# Maximum capacity depending on selected pipette
pipette_max_volume = 20 if pipette == p20 else 200
print(f"📏 Max pipette capacity: {pipette_max_volume} µL")

# Number of pipetting operations required
number_of_pipetting_operation = math.ceil(volume / pipette_max_volume)
print(f"🔁 Number of pipetting operations: {number_of_pipetting_operation}")

# Volume for each pipetting operation
tip_volume = volume / number_of_pipetting_operation
print(f"📦 Volume per operation: {tip_volume:.2f} µL")

# Apply the speed to the selected pipette
pipette.flow_rate.aspirate = pipetting_speed
pipette.flow_rate.dispense = pipetting_speed
pipette.flow_rate.blow_out = pipetting_speed

# Display applied speeds
print(f"⚙️ Flow rate settings for pipette {'p20' if pipette == p20 else 'p300'}:")
print(f"  • Aspirate speed: {pipette.flow_rate.aspirate} µL/sec")
print(f"  • Dispense speed: {pipette.flow_rate.dispense} µL/sec")
print(f"  • Blow out speed: {pipette.flow_rate.blow_out} µL/sec")

#__________________________________________________________________________________________
#|RUN PROTOCOL|#

start_time = time.time() # Start of the timer

try:
    with open(step1p2_gga_master_mix_enzyme, 'r') as file:
        reader = csv.DictReader(file)
        rows = list(reader)
        if not rows:
            print("❌ Error: CSV file is empty or has no data rows.")
        else:
            print("✅ CSV successfully read!")

            for row in rows:
                src_well       = row['source']
                dst_well       = row['destination']
                lab_src_name   = row['source_labware']
                lab_dst_name   = row['destination_labware']

                # → here we go from the chain to the container
                lab_src = labware_map[lab_src_name]
                lab_dst = labware_map[lab_dst_name]

                # MIXING SOURCE
                pipette.default_speed = speed_outside_labware
                pipette.pick_up_tip()
                pipette.move_to(lab_src[src_well].top(z=top_z_blow_out_microtube))                
                pipette.default_speed = speed_inside_labware
                pipette.move_to(lab_src[src_well].bottom(z=bottom_z_microtube))                        
                # Manual mixing without Opentrons 'mix' command
                for cycle in range(3):
                    pipette.aspirate(tip_volume)
                    pipette.dispense(tip_volume)
                    ctx.delay(seconds=delay_viscous)
                pipette.default_speed = speed_inside_labware_viscous   # Exit slowly from the liquid
                pipette.move_to(lab_src[src_well].top(z=top_z_blow_out_microtube))
                ctx.delay(seconds=delay)   # Before blow out to allow liquid to settle
                # Blow out and clean pipette tip
                pipette.blow_out()
                pipette.default_speed = speed_inside_labware
                pipette.move_to(lab_src[src_well].bottom(z=bottom_z_with_liquid_microtube))   # Like a 'touch tip' inside liquid: avoid to have to much liquid lost
                pipette.default_speed = speed_inside_labware_viscous   # Exit slowly from the liquid
                pipette.move_to(lab_src[src_well].top(z=top_z_blow_out_microtube))                
                # End of step
                pipette.default_speed = speed_outside_labware
                pipette.drop_tip()
                increment_tip_count(ctx, pipette)                   
 
                for _ in range(number_of_pipetting_operation):         

                    # Update total tip_volumes
                    update_volume(lab_src_name, src_well, -tip_volume)  # Subtract from source well
                    update_volume(lab_dst_name, dst_well, tip_volume)   # Add to destination well                 

                    # Change tip between operation
                    pipette.default_speed = speed_outside_labware
                    pipette.pick_up_tip()
                    
                    # ASPIRATE FROM SOURCE
                    pipette.move_to(lab_src[src_well].top(z=top_z_blow_out_microtube))
                    pipette.default_speed = speed_inside_labware
                    pipette.move_to(lab_src[src_well].bottom(z=bottom_z_microtube))
                    pipette.aspirate(tip_volume)
                    ctx.delay(seconds=delay_viscous)
                    pipette.default_speed = speed_inside_labware_viscous   # Exit slowly from the liquid
                    pipette.move_to(lab_src[src_well].top(z=top_z_blow_out_microtube))
                    pipette.default_speed = speed_outside_labware 
            
                    # DISPENSE TO DESTINATION                                
                    # Dispense
                    pipette.move_to(lab_dst[dst_well].top(z=bottom_z_with_liquid_microtube))   # Liquid already present in the tube
                    pipette.default_speed = speed_inside_labware 
                    pipette.move_to(lab_dst[dst_well].bottom(z=bottom_z_microtube))
                    pipette.dispense(tip_volume, push_out=push_out)   # No mixing: this is the first thing added to the tube
                    ctx.delay(seconds=delay_viscous)
                    # Blow out and clean pipette tip
                    pipette.default_speed = speed_inside_labware_viscous   # Exit slowly from the liquid
                    pipette.move_to(lab_dst[dst_well].top(z=top_z_blow_out_microtube))
                    ctx.delay(seconds=delay)   # Before blow out to allow liquid to settle
                    pipette.blow_out()
                    pipette.default_speed = speed_inside_labware                    
                    pipette.move_to(lab_dst[dst_well].bottom(z=bottom_z_with_liquid_microtube))   # Like a 'touch tip' inside liquid: avoid to have to much liquid lost
                    pipette.default_speed = speed_inside_labware_viscous   # Exit slowly from the liquid
                    pipette.move_to(lab_dst[dst_well].top(z=top_z_blow_out_microtube))
                    # End of step
                    pipette.default_speed = speed_outside_labware    
                    pipette.drop_tip()
                    increment_tip_count(ctx, pipette)

except FileNotFoundError:
    print(f"❌ Error: The file '{step1p2_gga_master_mix_enzyme}' cannot be found.")
except Exception as e:
    print(f"❌ Unexpected error: {e}")

step1p2_time = time.time() - start_time # Calculation of elapsed time
ctx.comment(f"🤖 Step 1.2 Golden Gate Assembly Master Mix preparation: NEBridge® Golden Gate Assembly addition completed in {step1p2_time:.2f} seconds.")

# Displays
for instruction in ctx.commands()[start_index:]: # Display only the commands executed in this block
    print(instruction)
display_final_volumes() # Display summary of final volumes
display_tip_usage() # Display number of tips used

# 🤖 Step 1.2 completed in 3 minutes 2 seconds using 2 P300 tips

In [ ]:
##|--STEP 1.3 PREPARE GOLDEN-GATE-ASSEMBLY W/O EBLOCKS MASTER MIX (HIGH VOLUME AND LOW VISCOSITY FIRST): mixing master mix--|##

# Display only the new instructions generated in this block code, without displaying those that have been executed previously
start_index = len(ctx.commands())

# Reset the total volumes and maximum capacities of labwares
total_volumes = {'aluminumblock_nest_96_wellplate_100ul_pos03_from_fridge_slot01': {}, 'tc_plate': {}, 'aluminumblock_nest_24_500uL_pos09': {}}
max_volumes = {'aluminumblock_nest_96_wellplate_100ul_pos03_from_fridge_slot01': 100, 'tc_plate': 100, 'aluminumblock_nest_24_500uL_pos09': 2000}

# Source(s) and destination(s)
print_wells_from_csv(step1p3_gga_master_mix_mixing)

# Define the volume to be mixed according to the protocol conditions
volume = (t4_dna_ligase_buffer_10x + nebridge_gga) * n_bigblocks * n_replicas_gga * overage_factor_during_splitting_in_mini_master_mix_step
print(f"🔢 Total volume to mix (Golden Gate Assembly master mix) = {volume:.2f} µL")

# Determine if the liquid is viscous
viscous_flag = True
print(f"💧 Is the liquid viscous? {'Yes' if viscous_flag else 'No'}")

# Pipette and corresponding speed selection
pipette, pipetting_speed, push_out = choose_pipette_and_speed(volume, viscous=viscous_flag)
print(f"🧪 Selected pipette: {'p20' if pipette == p20 else 'p300'}")
print(f"🚀 Pipetting speed: {pipetting_speed} µL/sec")

# Maximum capacity depending on selected pipette
pipette_max_volume = 20 if pipette == p20 else 200
print(f"📏 Max pipette capacity: {pipette_max_volume} µL")

# Number of pipetting operations required
number_of_pipetting_operation = math.ceil(volume / pipette_max_volume)
print(f"🔁 Number of pipetting operations: {number_of_pipetting_operation}")

# Volume for each pipetting operation
tip_volume = volume / number_of_pipetting_operation
print(f"📦 Volume per operation: {tip_volume:.2f} µL")

# Apply the speed to the selected pipette
pipette.flow_rate.aspirate = pipetting_speed
pipette.flow_rate.dispense = pipetting_speed
pipette.flow_rate.blow_out = pipetting_speed

# Display applied speeds
print(f"⚙️ Flow rate settings for pipette {'p20' if pipette == p20 else 'p300'}:")
print(f"  • Aspirate speed: {pipette.flow_rate.aspirate} µL/sec")
print(f"  • Dispense speed: {pipette.flow_rate.dispense} µL/sec")
print(f"  • Blow out speed: {pipette.flow_rate.blow_out} µL/sec")

#__________________________________________________________________________________________
#|RUN PROTOCOL|#

start_time = time.time() # Start of the timer

try:
    with open(step1p3_gga_master_mix_mixing, 'r') as file:
        reader = csv.DictReader(file)
        rows = list(reader)
        if not rows:
            print("❌ Error: CSV file is empty or has no data rows.")
        else:
            print("✅ CSV successfully read!")

            for row in rows:
                src_well       = row['source']
                dst_well       = row['destination']
                lab_src_name   = row['source_labware']
                lab_dst_name   = row['destination_labware']

                # → here we go from the chain to the container
                lab_src = labware_map[lab_src_name]
                lab_dst = labware_map[lab_dst_name]           
 
                for _ in range(number_of_pipetting_operation):                       
                    
                    # MIXING SOURCE
                    pipette.default_speed = speed_outside_labware
                    pipette.pick_up_tip()
                    pipette.move_to(lab_src[src_well].top(z=top_z_blow_out_microtube))
                    pipette.default_speed = speed_inside_labware       
                    pipette.move_to(lab_src[src_well].bottom(z=bottom_z_with_liquid_microtube))   # Liquid in tube     
                    # Manual mixing without Opentrons 'mix' command
                    for cycle in range(5):
                        pipette.aspirate(tip_volume)
                        pipette.dispense(tip_volume)   # No push out when mixing
                        ctx.delay(seconds=delay_viscous)   
                    # Blow out and clean pipette tip
                    pipette.default_speed = speed_inside_labware_viscous   # Exit slowly from the liquid
                    pipette.move_to(lab_src[src_well].top(z=top_z_blow_out_microtube))
                    ctx.delay(seconds=delay)   # Before blow out to allow liquid to settle
                    pipette.blow_out()
                    pipette.default_speed = speed_inside_labware
                    pipette.move_to(lab_dst[dst_well].bottom(z=bottom_z_with_liquid_microtube))   # Like a 'touch tip' inside liquid: avoid to have to much liquid lost
                    pipette.default_speed = speed_inside_labware_viscous   # Exit slowly from the liquid                    
                    pipette.move_to(lab_src[src_well].top(z=top_z_blow_out_microtube))
                    # End of step
                    pipette.default_speed = speed_outside_labware
                    pipette.drop_tip()
                    increment_tip_count(ctx, pipette)    

except FileNotFoundError:
    print(f"❌ Error: The file '{step1p3_gga_master_mix_mixing}' cannot be found.")
except Exception as e:
    print(f"❌ Unexpected error: {e}")

step1p3_time = time.time() - start_time # Calculation of elapsed time
ctx.comment(f"🤖 Step 1.3 Golden Gate Assembly Master Mix preparation: mixing completed in {step1p3_time:.2f} seconds.")

# Displays
for instruction in ctx.commands()[start_index:]: # Display only the commands executed in this block
    print(instruction)
display_final_volumes() # Display summary of final volumes
display_tip_usage() # Display number of tips used

# 🤖 Step 1.3 completed in 2 minutes 38 seconds using 1 P300 tips

<hr style="height: 2px; background-color: black; border: none;">

## **🤖 | STEP 2 - OPENTRONS OT-2 ROBOT** | 🤖
### Distribute Golden Gate Master Mix w/o eBlocks

***

In [ ]:
##|--STEP 2. SPLIT GOLDEN-GATE-ASSEMBLY MASTER MIX IN N MINI MASTER-MIX--|##

# Display only the new instructions generated in this block code, without displaying those that have been executed previously
start_index = len(ctx.commands())

# Reset the total volumes and maximum capacities of labwares
total_volumes = {'aluminumblock_nest_96_wellplate_100ul_pos03_from_fridge_slot01': {}, 'tc_plate': {}, 'aluminumblock_nest_24_500uL_pos09': {}}
max_volumes = {'aluminumblock_nest_96_wellplate_100ul_pos03_from_fridge_slot01': 100, 'tc_plate': 100, 'aluminumblock_nest_24_500uL_pos09': 2000}

# Source(s) and destination(s)
print_wells_from_csv(step2_gga_master_mix_mixing)

# Define the volume to be transferred according to the protocol conditions
volume = (gga_final_reaction_volume - all_eblocks_raw_idt) * n_replicas_gga * overage_factor_during_splitting_in_mini_master_mix_step
print(f"🔢 Total volume to transfer (Golden Gate Assembly Mix w/o eBlocks) = {volume:.2f} µL")

# Determine if the liquid is viscous
viscous_flag = True
print(f"💧 Is the liquid viscous? {'Yes' if viscous_flag else 'No'}")

# Pipette and corresponding speed selection
pipette, pipetting_speed, push_out = choose_pipette_and_speed(volume, viscous=viscous_flag)
print(f"🧪 Selected pipette: {'p20' if pipette == p20 else 'p300'}")
print(f"🚀 Pipetting speed: {pipetting_speed} µL/sec")

# Maximum capacity depending on selected pipette
pipette_max_volume = 20 if pipette == p20 else 200
print(f"📏 Max pipette capacity: {pipette_max_volume} µL")

# Number of pipetting operations required
number_of_pipetting_operation = math.ceil(volume / pipette_max_volume)
print(f"🔁 Number of pipetting operations: {number_of_pipetting_operation}")

# Volume for each pipetting operation
tip_volume = volume / number_of_pipetting_operation
print(f"📦 Volume per operation: {tip_volume:.2f} µL")

# Apply the speed to the selected pipette
pipette.flow_rate.aspirate = pipetting_speed
pipette.flow_rate.dispense = pipetting_speed
pipette.flow_rate.blow_out = pipetting_speed

# Display applied speeds
print(f"⚙️ Flow rate settings for pipette {'p20' if pipette == p20 else 'p300'}:")
print(f"  • Aspirate speed: {pipette.flow_rate.aspirate} µL/sec")
print(f"  • Dispense speed: {pipette.flow_rate.dispense} µL/sec")
print(f"  • Blow out speed: {pipette.flow_rate.blow_out} µL/sec")

#__________________________________________________________________________________________
#|RUN PROTOCOL|#

start_time = time.time() # Start of the timer
ctx.home()   # Important to keep the calibration between the steps

try:
    with open(step2_gga_master_mix_mixing, 'r') as file:
        reader = csv.DictReader(file)
        rows = list(reader)
        if not rows:
            print("❌ Error: CSV file is empty or has no data rows.")
        else:
            print("✅ CSV successfully read!")

            random.shuffle(rows) # apply a random shuffle function on the rows

            for row in rows:
                src_well       = row['source']
                dst_well       = row['destination']
                lab_src_name   = row['source_labware']
                lab_dst_name   = row['destination_labware']

                # → here we go from the chain to the container
                lab_src = labware_map[lab_src_name]
                lab_dst = labware_map[lab_dst_name]                  
 
                for _ in range(number_of_pipetting_operation):         

                    # Update total tip_volumes
                    update_volume(lab_src_name, src_well, -tip_volume)  # Subtract from source well
                    update_volume(lab_dst_name, dst_well, tip_volume)   # Add to destination well                 

                    # Change tip between operation
                    pipette.default_speed = speed_outside_labware
                    pipette.pick_up_tip()
                    
                    # ASPIRATE FROM SOURCE
                    pipette.move_to(lab_src[src_well].top(z=top_z_blow_out_microtube))
                    pipette.default_speed = speed_inside_labware      
                    pipette.move_to(lab_src[src_well].bottom(z=bottom_z_microtube))
                    pipette.aspirate(tip_volume)
                    ctx.delay(seconds=delay_viscous)
                    pipette.default_speed = speed_inside_labware_viscous   # Exit slowly from the liquid
                    pipette.move_to(lab_src[src_well].top(z=top_z_blow_out_microtube)) 
                    pipette.default_speed = speed_outside_labware 
            
                    # DISPENSE TO DESTINATION
                    # Dispense
                    pipette.move_to(lab_dst[dst_well].top(z=top_z_blow_out_microtube))
                    pipette.default_speed = speed_inside_labware              
                    pipette.move_to(lab_dst[dst_well].bottom(z=bottom_z_microtube))
                    pipette.dispense(tip_volume, push_out=push_out)   # No mixing: this is the first thing added to the tube
                    ctx.delay(seconds=delay)                   
                    # Blow out and clean pipette tip
                    pipette.default_speed = speed_inside_labware_viscous   # Exit slowly from the liquid
                    pipette.move_to(lab_dst[dst_well].top(z=top_z_blow_out_microtube))
                    ctx.delay(seconds=delay)   # Before blow out to allow liquid to settle
                    pipette.blow_out()   # Blow out any remaining liquid
                    pipette.default_speed = speed_inside_labware
                    pipette.move_to(lab_dst[dst_well].bottom(z=bottom_z_with_liquid_microtube))   # Like a 'touch tip' inside liquid: avoid to have to much liquid lost
                    pipette.default_speed = speed_inside_labware_viscous   # Exit slowly from the liquid
                    pipette.move_to(lab_dst[dst_well].top(z=top_z_blow_out_microtube))                                     
                    # End of step
                    pipette.default_speed = speed_outside_labware 
                    pipette.drop_tip()
                    increment_tip_count(ctx, pipette)                               
                    
except FileNotFoundError:
    print(f"❌ Error: The file '{step2_gga_master_mix_mixing}' cannot be found.")
except Exception as e:
    print(f"❌ Unexpected error: {e}")

step2_time = time.time() - start_time # Calculation of elapsed time
ctx.comment(f"🤖 Step 2. Golden Gate Assembly master mix w/o eBlocks splitting in mini master mix completed {step2_time:.2f} seconds.")

# Displays
for instruction in ctx.commands()[start_index:]: # Display only the commands executed in this block
    print(instruction)
display_final_volumes() # Display summary of final volumes
display_tip_usage() # Display number of tips used

#🤖 Step 2. completed in 12 minutes 21 seconds using 12 P20 tips

<hr style="height: 2px; background-color: black; border: none;">

## **🦾 | STEP 3 - DOBOT ARM & SLIDING RAIL | 🦾**
### Bring the 96-well plate containing Golden Gate and PCR reagents from the fridge
***

In [ ]:
start_time = time.time()  # start of the timer
time.sleep(5)  # wait 5 seconds: be ready to press the emergency stop button
arm.mov_arm_coords_and_rail(*ws.pos_security)  # in case it isn't yet
ws.do_action_slot(slot="fridge_slot1", griper_action=arm.close_gripper)
ws.do_action_slot(slot="temp_module_slot3", griper_action=arm.open_gripper)
step3_time = time.time() - start_time # Calculation of elapsed time

print(f"🦾 Step 3 completed in {step3_time:.2f} seconds.")

<hr style="height: 2px; background-color: black; border: none;">

## 🤖 | **STEP 4 - OPENTRONS OT-2 ROBOT** | 🤖
### Add eBlocks corresponding to each BigBlock in Golden Gate mini Master-Mix

***

In [ ]:
##|--STEP 4.1. ADD EBLOCKS TO GOLDEN GATE MINI MASTER MIX--|##

# Display only the new instructions generated in this block code, without displaying those that have been executed previously
start_index = len(ctx.commands())

# Reset the total volumes and maximum capacities of labwares
total_volumes = {'aluminumblock_nest_96_wellplate_100ul_pos03_from_fridge_slot01': {}, 'tc_plate': {}, 'aluminumblock_nest_24_500uL_pos09': {}}
max_volumes = {'aluminumblock_nest_96_wellplate_100ul_pos03_from_fridge_slot01': 100, 'tc_plate': 100, 'aluminumblock_nest_24_500uL_pos09': 2000}

# Source(s) and destination(s)
print_wells_from_csv(step4p1_adding_eblocks)

# Define the volume to be transferred according to the protocol conditions
volume = eBlock_raw_idt * n_replicas_gga * overage_factor_during_splitting_in_mini_master_mix_step
print(f"🔢 Total volume to transfer (raw eBlocks IDT at 10 ng/µL) = {volume:.2f} µL")

# Determine if the liquid is viscous
viscous_flag = True
print(f"💧 Is the liquid viscous? {'Yes' if viscous_flag else 'No'}")

# Pipette and corresponding speed selection
pipette, pipetting_speed, push_out = choose_pipette_and_speed(volume, viscous=viscous_flag)
print(f"🧪 Selected pipette: {'p20' if pipette == p20 else 'p300'}")
print(f"🚀 Pipetting speed: {pipetting_speed} µL/sec")

# Maximum capacity depending on selected pipette
pipette_max_volume = 20 if pipette == p20 else 200
print(f"📏 Max pipette capacity: {pipette_max_volume} µL")

# Number of pipetting operations required
number_of_pipetting_operation = math.ceil(volume / pipette_max_volume)
print(f"🔁 Number of pipetting operations: {number_of_pipetting_operation}")

# Volume for each pipetting operation
tip_volume = volume / number_of_pipetting_operation
print(f"📦 Volume per operation: {tip_volume:.2f} µL")

# Apply the speed to the selected pipette
pipette.flow_rate.aspirate = pipetting_speed
pipette.flow_rate.dispense = pipetting_speed
pipette.flow_rate.blow_out = pipetting_speed

# Display applied speeds
print(f"⚙️ Flow rate settings for pipette {'p20' if pipette == p20 else 'p300'}:")
print(f"  • Aspirate speed: {pipette.flow_rate.aspirate} µL/sec")
print(f"  • Dispense speed: {pipette.flow_rate.dispense} µL/sec")
print(f"  • Blow out speed: {pipette.flow_rate.blow_out} µL/sec")

#__________________________________________________________________________________________
#|RUN PROTOCOL|#

start_time = time.time() # Start of the timer
ctx.home()   # Important to keep the calibration between the steps

try:
    with open(step4p1_adding_eblocks, 'r') as file:
        reader = csv.DictReader(file)
        rows = list(reader)
        if not rows:
            print("❌ Error: CSV file is empty or has no data rows.")
        else:
            print("✅ CSV successfully read!")

            random.shuffle(rows) # apply a random shuffle function on the rows

            for row in rows:
                src_well       = row['source']
                dst_well       = row['destination']
                lab_src_name   = row['source_labware']
                lab_dst_name   = row['destination_labware']

                # → here we go from the chain to the container
                lab_src = labware_map[lab_src_name]
                lab_dst = labware_map[lab_dst_name]               
 
                for _ in range(number_of_pipetting_operation):         

                    # Update total tip_volumes
                    update_volume(lab_src_name, src_well, -tip_volume)  # Subtract from source well
                    update_volume(lab_dst_name, dst_well, tip_volume)   # Add to destination well                 

                    # Change tip between operation
                    pipette.default_speed = speed_outside_labware
                    pipette.pick_up_tip()
                    
                    # ASPIRATE FROM SOURCE
                    pipette.move_to(lab_src[src_well].top(z=top_z_slit_seal_microwell))
                    pipette.default_speed = speed_inside_labware                                                      
                    pipette.move_to(lab_src[src_well].bottom(z=bottom_z_microwell))
                    # Manual mixing without Opentrons 'mix' command
                        # Don't need to do a lot of mixing: will aspirate almost all the contents of the well for the transfer operation
                    for cycle in range(1):
                        pipette.aspirate(tip_volume)
                        pipette.dispense(tip_volume)   # No push out when mixing
                        ctx.delay(seconds=delay)                   
                    pipette.aspirate(tip_volume)
                    ctx.delay(seconds=delay)
                    pipette.move_to(lab_src[src_well].top(z=top_z_slit_seal_microwell))
                    pipette.default_speed = speed_outside_labware 
            
                    # DISPENSE TO DESTINATION
                    # Dispense
                    pipette.move_to(lab_dst[dst_well].top(z=top_z_blow_out_microtube))
                    pipette.default_speed = speed_inside_labware
                    pipette.move_to(lab_dst[dst_well].bottom(z=bottom_z_with_liquid_microtube))
                    pipette.dispense(tip_volume)
                    # Wash tip with one mixing operation
                    for cycle in range(1):
                        pipette.aspirate(tip_volume)
                        pipette.dispense(tip_volume)   # No push out when mixing
                        ctx.delay(seconds=delay)
                    # Blow out and clean pipette tip
                    pipette.move_to(lab_dst[dst_well].top(z=top_z_blow_out_microtube))
                    ctx.delay(seconds=delay)   # Before blow out to allow liquid to settle
                    pipette.blow_out()   # Blow out any remaining liquid
                    pipette.move_to(lab_dst[dst_well].bottom(z=bottom_z_with_liquid_microtube))   # Like a 'touch tip' inside liquid: avoid to have to much liquid lost
                    pipette.move_to(lab_dst[dst_well].top(z=top_z_blow_out_microtube))   
                    # End of step
                    pipette.default_speed = speed_outside_labware 
                    pipette.drop_tip()
                    increment_tip_count(ctx, pipette)

except FileNotFoundError:
    print(f"❌ Error: The file '{step4p1_adding_eblocks}' cannot be found.")
except Exception as e:
    print(f"❌ Unexpected error: {e}")

step4_time = time.time() - start_time # Calculation of elapsed time
ctx.comment(f"🤖 Step 4.1. Addition of eBlocks to Golden Gate Assembly mini Master Mix completed in {step4_time:.2f} seconds.")

# Displays
for instruction in ctx.commands()[start_index:]: # Display only the commands executed in this block
    print(instruction)
display_final_volumes() # Display summary of final volumes
display_tip_usage() # Display number of tips used

#🤖 Step 4.1 completed in 71 minutes 45 seconds using 96 P20 tips 

In [ ]:
##|--STEP 4.2. MIX GOLDEN GATE MINI MASTER MIX WITH EBLOCKS--|##

# Display only the new instructions generated in this block code, without displaying those that have been executed previously
start_index = len(ctx.commands())

# Reset the total volumes and maximum capacities of labwares
total_volumes = {'aluminumblock_nest_96_wellplate_100ul_pos03_from_fridge_slot01': {}, 'tc_plate': {}, 'aluminumblock_nest_24_500uL_pos09': {}}
max_volumes = {'aluminumblock_nest_96_wellplate_100ul_pos03_from_fridge_slot01': 100, 'tc_plate': 100, 'aluminumblock_nest_24_500uL_pos09': 2000}

# Source(s) and destination(s)
print_wells_from_csv(step4p2_mixing)

# Define the volume to be mixed according to the protocol conditions
volume = gga_final_reaction_volume * n_replicas_gga * overage_factor_in_tc
print(f"🔢 Total volume to mix (Golden Gate Assembly master mix) = {volume:.2f} µL")

# Determine if the liquid is viscous
viscous_flag = False
print(f"💧 Is the liquid viscous? {'Yes' if viscous_flag else 'No'}")

# Pipette and corresponding speed selection
pipette, pipetting_speed, push_out = choose_pipette_and_speed(volume, viscous=viscous_flag)
print(f"🧪 Selected pipette: {'p20' if pipette == p20 else 'p300'}")
print(f"🚀 Pipetting speed: {pipetting_speed} µL/sec")

# Maximum capacity depending on selected pipette
pipette_max_volume = 20 if pipette == p20 else 200
print(f"📏 Max pipette capacity: {pipette_max_volume} µL")

# Number of pipetting operations required
number_of_pipetting_operation = math.ceil(volume / pipette_max_volume)
print(f"🔁 Number of pipetting operations: {number_of_pipetting_operation}")

# Volume for each pipetting operation
tip_volume = volume / number_of_pipetting_operation
print(f"📦 Volume per operation: {tip_volume:.2f} µL")

# Apply the speed to the selected pipette
pipette.flow_rate.aspirate = pipetting_speed
pipette.flow_rate.dispense = pipetting_speed
pipette.flow_rate.blow_out = pipetting_speed

# Display applied speeds
print(f"⚙️ Flow rate settings for pipette {'p20' if pipette == p20 else 'p300'}:")
print(f"  • Aspirate speed: {pipette.flow_rate.aspirate} µL/sec")
print(f"  • Dispense speed: {pipette.flow_rate.dispense} µL/sec")
print(f"  • Blow out speed: {pipette.flow_rate.blow_out} µL/sec")

#__________________________________________________________________________________________
#|RUN PROTOCOL|#

start_time = time.time() # Start of the timer
ctx.home()   # Important to keep the calibration between the steps

try:
    with open(step4p2_mixing, 'r') as file:
        reader = csv.DictReader(file)
        rows = list(reader)
        if not rows:
            print("❌ Error: CSV file is empty or has no data rows.")
        else:
            print("✅ CSV successfully read!")

            random.shuffle(rows) # apply a random shuffle function on the rows

            for row in rows:
                src_well       = row['source']
                dst_well       = row['destination']
                lab_src_name   = row['source_labware']
                lab_dst_name   = row['destination_labware']

                # → here we go from the chain to the container
                lab_src = labware_map[lab_src_name]
                lab_dst = labware_map[lab_dst_name]           
 
                for _ in range(number_of_pipetting_operation):                       
                    
                    # MIXING SOURCE
                    pipette.default_speed = speed_outside_labware
                    pipette.pick_up_tip()
                    pipette.move_to(lab_dst[dst_well].top(z=top_z_blow_out_microtube))                                                        
                    pipette.default_speed = speed_inside_labware
                    pipette.move_to(lab_src[src_well].bottom(z=bottom_z_microtube))        
                    # Manual mixing without Opentrons 'mix' command
                    for cycle in range(5):
                        pipette.aspirate(tip_volume)
                        pipette.dispense(tip_volume)   # No push out when mixing
                        ctx.delay(seconds=delay)                                           
                    # Blow out and clean pipette tip
                    pipette.move_to(lab_dst[dst_well].top(z=top_z_blow_out_microtube))
                    ctx.delay(seconds=delay)   # Before blow out to allow liquid to settle
                    pipette.blow_out()   # Blow out any remaining liquid
                    pipette.move_to(lab_dst[dst_well].bottom(z=bottom_z_with_liquid_microtube))   # Like a 'touch tip' inside liquid: avoid to have to much liquid lost
                    pipette.move_to(lab_dst[dst_well].top(z=top_z_blow_out_microtube))   
                    # End of step
                    pipette.default_speed = speed_outside_labware 
                    pipette.drop_tip()
                    increment_tip_count(ctx, pipette)  

except FileNotFoundError:
    print(f"❌ Error: The file '{step4p2_mixing}' cannot be found.")
except Exception as e:
    print(f"❌ Unexpected error: {e}")

step4p2_time = time.time() - start_time # Calculation of elapsed time
ctx.comment(f"🤖 Step 4.2 Mixing Golden Gate Assembly mini Master Mix with eBlocks completed in {step4p2_time:.2f} seconds.")

# Displays
for instruction in ctx.commands()[start_index:]: # Display only the commands executed in this block
    print(instruction)
display_final_volumes() # Display summary of final volumes
display_tip_usage() # Display number of tips used

#🤖 Step 4.2 completed in 12 minutes using 12 P20 tips

<hr style="height: 2px; background-color: black; border: none;">

## 🦾 | **STEP 5 - DOBOT ARM & SLIDING RAIL** | 🦾
### Store the 96-well plate containing eBlocks from OT-2 to the fridge and bring an empty 96-well plate from the fridge to OT-2

***

In [ ]:
start_time = time.time() 
time.sleep(5)
arm.mov_arm_coords_and_rail(*ws.pos_security)
ws.do_action_slot(slot="temp_module_slot3", griper_action=arm.close_gripper)
ws.do_action_slot(slot="fridge_slot1", griper_action=arm.open_gripper)
ws.do_action_slot(slot="fridge_slot2", griper_action=arm.close_gripper)
ws.do_action_slot(slot="temp_module_slot3", griper_action=arm.open_gripper)
step5_time = time.time() - start_time # Calculation of elapsed time

print(f"🦾 Step 5 completed in {step5_time:.2f} seconds.")

<hr style="height: 2px; background-color: black; border: none;">

## 🤖 | **STEP 6 - OPENTRONS OT-2 ROBOT** | 🤖
### Transfer Golden Gate Assembly mini master mix with eBlocks to the thermocycler and launch the reaction

***


In [ ]:
# Open themocycler lid and cool down the block
tc_mod.open_lid()
tc_mod.set_block_temperature(temperature=4)

In [ ]:
##|--STEP 6.1. TRANSFER THE GOLDEN GATE MINI MASTER MIX WITH eBLOCKS TO THE THERMOCYCLER--|##

# Display only the new instructions generated in this block code, without displaying those that have been executed previously
start_index = len(ctx.commands())

# Reset the total volumes and maximum capacities of labwares
total_volumes = {'aluminumblock_nest_96_wellplate_100ul_pos03_from_fridge_slot01': {}, 'tc_plate': {}, 'aluminumblock_nest_24_500uL_pos09': {}}
max_volumes = {'aluminumblock_nest_96_wellplate_100ul_pos03_from_fridge_slot01': 100, 'tc_plate': 100, 'aluminumblock_nest_24_500uL_pos09': 2000}

# Source(s) and destination(s)
print_wells_from_csv(step6p1_gga_transfer_to_tc)

# Define the volume to be mixed according to the protocol conditions
volume = gga_final_reaction_volume * n_replicas_gga * overage_factor_in_tc
print(f"🔢 Total volume to mix (Golden Gate Assembly master mix) = {volume:.2f} µL")

# Determine if the liquid is viscous
viscous_flag = False
print(f"💧 Is the liquid viscous? {'Yes' if viscous_flag else 'No'}")

# Pipette and corresponding speed selection
pipette, pipetting_speed, push_out = choose_pipette_and_speed(volume, viscous=viscous_flag)
print(f"🧪 Selected pipette: {'p20' if pipette == p20 else 'p300'}")
print(f"🚀 Pipetting speed: {pipetting_speed} µL/sec")

# Maximum capacity depending on selected pipette
pipette_max_volume = 20 if pipette == p20 else 200
print(f"📏 Max pipette capacity: {pipette_max_volume} µL")

# Number of pipetting operations required
number_of_pipetting_operation = math.ceil(volume / pipette_max_volume)
print(f"🔁 Number of pipetting operations: {number_of_pipetting_operation}")

# Volume for each pipetting operation
tip_volume = volume / number_of_pipetting_operation
print(f"📦 Volume per operation: {tip_volume:.2f} µL")

# Apply the speed to the selected pipette
pipette.flow_rate.aspirate = pipetting_speed
pipette.flow_rate.dispense = pipetting_speed
pipette.flow_rate.blow_out = pipetting_speed

# Display applied speeds
print(f"⚙️ Flow rate settings for pipette {'p20' if pipette == p20 else 'p300'}:")
print(f"  • Aspirate speed: {pipette.flow_rate.aspirate} µL/sec")
print(f"  • Dispense speed: {pipette.flow_rate.dispense} µL/sec")
print(f"  • Blow out speed: {pipette.flow_rate.blow_out} µL/sec")

#__________________________________________________________________________________________
#|RUN PROTOCOL|#

start_time = time.time() # Start of the timer
ctx.home()   # Important to keep the calibration between the steps

try:
    with open(step6p1_gga_transfer_to_tc, 'r') as file:
        reader = csv.DictReader(file)
        rows = list(reader)
        if not rows:
            print("❌ Error: CSV file is empty or has no data rows.")
        else:
            print("✅ CSV successfully read!")

            random.shuffle(rows) # apply a random shuffle function on the rows

            for row in rows:
                src_well       = row['source']
                dst_well       = row['destination']
                lab_src_name   = row['source_labware']
                lab_dst_name   = row['destination_labware']

                # → here we go from the chain to the container
                lab_src = labware_map[lab_src_name]
                lab_dst = labware_map[lab_dst_name]           
 
                for _ in range(number_of_pipetting_operation):         

                    # Update total tip_volumes
                    update_volume(lab_src_name, src_well, -tip_volume)  # Subtract from source well
                    update_volume(lab_dst_name, dst_well, tip_volume)   # Add to destination well                 

                    # Change tip between operation
                    pipette.default_speed = speed_outside_labware
                    pipette.pick_up_tip()
                    
                    # ASPIRATE FROM SOURCE
                    pipette.move_to(lab_src[src_well].top(z=top_z_blow_out_microtube))
                    pipette.default_speed = speed_inside_labware
                    pipette.move_to(lab_src[src_well].bottom(z=bottom_z_microtube))                  
                    pipette.aspirate(tip_volume)  # Already mixed at step 4.2
                    ctx.delay(seconds=delay)
                    pipette.move_to(lab_src[src_well].top(z=top_z_blow_out_microtube))
                    pipette.default_speed = speed_outside_labware 
            
                    # DISPENSE TO DESTINATION
                    # Dispense
                    pipette.move_to(lab_dst[dst_well].bottom(z=bottom_z_microwell))   # Empty wells w/o seal
                    pipette.dispense(tip_volume)
                    # Blow out and clean pipette tip
                    pipette.default_speed = speed_inside_labware   # Not set yet for dispensed wells
                    pipette.move_to(lab_dst[dst_well].top(z=top_z_blow_out_microwell))
                    ctx.delay(seconds=delay)   # Before blow out to allow liquid to settle
                    pipette.blow_out()   # Blow out any remaining liquid
                    pipette.move_to(lab_dst[dst_well].bottom(z=bottom_z_with_liquid_microwell))   # Like a 'touch tip' inside liquid: avoid to have to much liquid lost
                    pipette.move_to(lab_dst[dst_well].top(z=top_z_blow_out_microwell))   
                    # End of step                  
                    pipette.default_speed = speed_outside_labware   # No seal                     
                    pipette.drop_tip()
                    increment_tip_count(ctx, pipette)

except FileNotFoundError:
    print(f"❌ Error: The file '{step6p1_gga_transfer_to_tc}' cannot be found.")
except Exception as e:
    print(f"❌ Unexpected error: {e}")

step6p1_time = time.time() - start_time # Calculation of elapsed time
ctx.comment(f"🤖 Step 6.1 Transfer the Golden Gate mini master mix with eblocks to the thermocycler completed in {step6p1_time:.2f} seconds.")

# Displays
for instruction in ctx.commands()[start_index:]: # Display only the commands executed in this block
    print(instruction)
display_final_volumes() # Display summary of final volumes
display_tip_usage() # Display number of tips used

#🤖 Step 6.1 completed in 7 minutes 41 seconds using 12 P20 tips

In [ ]:
# Shutdown temperature modules during thermocycler
temp_mod_pos03.deactivate()
temp_mod_pos09.deactivate()

In [ ]:
##|--STEP 6.2. RUN GOLDEN GATE IN THE THERMOCYCLER --|##

# Display only the new instructions generated in this block code, without displaying those that have been executed previously
start_index = len(ctx.commands())

#__________________________________________________________________________________________
#|RUN PROTOCOL|#

GGA_volume = gga_final_reaction_volume
print ("volume of GGA =", GGA_volume, "µL")

start_time = time.time() # Start of the timer

tc_mod.close_lid()
tc_block_volume = GGA_volume  # For precise temperature control, it is important to define the dispensing volume
tc_mod.set_lid_temperature(temperature=105)
for i in range(15):
    tc_mod.set_block_temperature(temperature=37, hold_time_minutes=5, block_max_volume=tc_block_volume)
    tc_mod.set_block_temperature(temperature=16, hold_time_minutes=5, block_max_volume=tc_block_volume)
tc_mod.set_block_temperature(temperature=60, hold_time_minutes=5, block_max_volume=tc_block_volume)
tc_mod.set_block_temperature(temperature=4)  # Keep block temperature at 4°C
tc_mod.set_lid_temperature(temperature=37)  # A part of products will remain in the thermocycler for PCR. Must be between 37°C and 110°C

step6p2_time = time.time() - start_time # Calculation of elapsed time
ctx.comment(f"🤖 Step 6.2 Golden Gate reaction in thermocycler completed in {step6p2_time:.2f} seconds.")

# Displays
for instruction in ctx.commands()[start_index:]: # Display only the commands executed in this block
    print(instruction)
display_final_volumes() # Display summary of final volumes
display_tip_usage() # Display number of tips used

#🤖 Step 6.1 completed in 2 hours 45 minutes

<hr style="height: 2px; background-color: black; border: none;">

## 🤖 | **STEP 7 - OPENTRONS OT-2 ROBOT** | 🤖
### Transfer BigBlocks from thermocycler to 96 wells plate on temperature module slot 03

***


In [ ]:
# Start temperature module
temp_mod_pos03.set_temperature(4)
# open thermocycler lid
tc_mod.open_lid()

In [ ]:
##|--STEP 7. TRANSFER BIGBLOCKS FROM THERMOCYCLER TO 96 WELLS PLATE ON TEMPERATURE MODULE SLOT 03--|##

# Display only the new instructions generated in this block code, without displaying those that have been executed previously
start_index = len(ctx.commands())

# Reset the total volumes and maximum capacities of labwares
total_volumes = {'aluminumblock_nest_96_wellplate_100ul_pos03_from_fridge_slot02': {}, 'tc_plate': {}, 'aluminumblock_nest_24_500uL_pos09': {}}
max_volumes = {'aluminumblock_nest_96_wellplate_100ul_pos03_from_fridge_slot02': 100, 'tc_plate': 100, 'aluminumblock_nest_24_500uL_pos09': 2000}

# Source(s) and destination(s)
print_wells_from_csv(step7p1_gga_transfer_from_tc)

# Define the volume to be mixed according to the protocol conditions
volume = gga_final_reaction_volume * n_replicas_gga * overage_factor_after_tc_run
print(f"🔢 Total volume to mix (Golden Gate Assembly master mix) = {volume:.2f} µL")

# Determine if the liquid is viscous
viscous_flag = False
print(f"💧 Is the liquid viscous? {'Yes' if viscous_flag else 'No'}")

# Pipette and corresponding speed selection
pipette, pipetting_speed, push_out = choose_pipette_and_speed(volume, viscous=viscous_flag)
print(f"🧪 Selected pipette: {'p20' if pipette == p20 else 'p300'}")
print(f"🚀 Pipetting speed: {pipetting_speed} µL/sec")

# Maximum capacity depending on selected pipette
pipette_max_volume = 20 if pipette == p20 else 200
print(f"📏 Max pipette capacity: {pipette_max_volume} µL")

# Number of pipetting operations required
number_of_pipetting_operation = math.ceil(volume / pipette_max_volume)
print(f"🔁 Number of pipetting operations: {number_of_pipetting_operation}")

# Volume for each pipetting operation
tip_volume = volume / number_of_pipetting_operation
print(f"📦 Volume per operation: {tip_volume:.2f} µL")

# Apply the speed to the selected pipette
pipette.flow_rate.aspirate = pipetting_speed
pipette.flow_rate.dispense = pipetting_speed
pipette.flow_rate.blow_out = pipetting_speed

# Display applied speeds
print(f"⚙️ Flow rate settings for pipette {'p20' if pipette == p20 else 'p300'}:")
print(f"  • Aspirate speed: {pipette.flow_rate.aspirate} µL/sec")
print(f"  • Dispense speed: {pipette.flow_rate.dispense} µL/sec")
print(f"  • Blow out speed: {pipette.flow_rate.blow_out} µL/sec")

#__________________________________________________________________________________________
#|RUN PROTOCOL|#

start_time = time.time() # Start of the timer
ctx.home()   # Important to keep the calibration between the steps

try:
    with open(step7p1_gga_transfer_from_tc, 'r') as file:
        reader = csv.DictReader(file)
        rows = list(reader)
        if not rows:
            print("❌ Error: CSV file is empty or has no data rows.")
        else:
            print("✅ CSV successfully read!")

            random.shuffle(rows) # apply a random shuffle function on the rows

            for row in rows:
                src_well       = row['source']
                dst_well       = row['destination']
                lab_src_name   = row['source_labware']
                lab_dst_name   = row['destination_labware']

                # → here we go from the chain to the container
                lab_src = labware_map[lab_src_name]
                lab_dst = labware_map[lab_dst_name]           
 
                for _ in range(number_of_pipetting_operation):         

                    # Update total tip_volumes
                    update_volume(lab_src_name, src_well, -tip_volume)  # Subtract from source well
                    update_volume(lab_dst_name, dst_well, tip_volume)   # Add to destination well                 

                    # Change tip between operation
                    pipette.default_speed = speed_outside_labware
                    pipette.pick_up_tip()
                    
                    # ASPIRATE FROM SOURCE
                    pipette.move_to(lab_src[src_well].top(z=top_z_blow_out_microwell))   # No seal
                    pipette.default_speed = speed_inside_labware
                    pipette.move_to(lab_src[src_well].bottom(z=bottom_z_microwell))                  
                    pipette.aspirate(tip_volume)
                    ctx.delay(seconds=delay)
                    pipette.move_to(lab_src[src_well].top(z=top_z_blow_out_microwell))
                    pipette.default_speed = speed_outside_labware 
            
                    # DISPENSE TO DESTINATION
                    # Dispense
                    pipette.move_to(lab_dst[dst_well].top(z=top_z_slit_seal_microwell))
                    pipette.default_speed = speed_inside_labware 
                    pipette.move_to(lab_dst[dst_well].bottom(z=bottom_z_microwell))   # Empty wells
                    pipette.dispense(tip_volume)
                    # Blow out and clean pipette tip
                    pipette.move_to(lab_dst[dst_well].top(z=top_z_blow_out_microwell))
                    ctx.delay(seconds=delay)   # Before blow out to allow liquid to settle
                    pipette.blow_out()   # Blow out any remaining liquid
                    pipette.move_to(lab_dst[dst_well].bottom(z=bottom_z_with_liquid_microwell))   # Like a 'touch tip' inside liquid: avoid to have to much liquid lost
                    pipette.move_to(lab_dst[dst_well].top(z=top_z_blow_out_microwell))   
                    # End of step 
                    pipette.move_to(lab_dst[dst_well].top(z=top_z_slit_seal_microwell))
                    pipette.default_speed = speed_outside_labware                     
                    pipette.drop_tip()
                    increment_tip_count(ctx, pipette)

except FileNotFoundError:
    print(f"❌ Error: The file '{step7p1_gga_transfer_from_tc}' cannot be found.")
except Exception as e:
    print(f"❌ Unexpected error: {e}")

step7_time = time.time() - start_time # Calculation of elapsed time
ctx.comment(f"🤖 Step 7. Transfer BigBlocks from thermocycler to 96 wells plate on Temperature module slot 03 completed in {step7_time:.2f} seconds.")

# Displays
for instruction in ctx.commands()[start_index:]: # Display only the commands executed in this block
    print(instruction)
display_final_volumes() # Display summary of final volumes
display_tip_usage() # Display number of tips used

#🤖 Step 7. completed in 8 minutes using 12 P20 tips

In [ ]:
# Shutdown temperature block and lid of the thermocycler
tc_mod.deactivate_block()
tc_mod.deactivate_lid()

<hr style="height: 2px; background-color: black; border: none;">

## 🦾 | **STEP 8 - DOBOT ARM & SLIDING RAIL** | 🦾
### Store the 96 wp containing the BigBlocks built during the Golden Gate Assembly to the fridge

***

In [ ]:
start_time = time.time() 
time.sleep(5)
arm.mov_arm_coords_and_rail(*ws.pos_security)
ws.do_action_slot(slot="temp_module_slot3", griper_action=arm.close_gripper)
ws.do_action_slot(slot="fridge_slot2", griper_action=arm.open_gripper)
step8_time = time.time() - start_time # Calculation of elapsed time

print(f"🦾 Step 8 completed in {step8_time:.2f} seconds.")

<hr style="height: 2px; background-color: black; border: none;">

# Restarting OT2 server

In [ ]:
#restarting the server so that you can use the app
os.system("systemctl start opentrons-robot-server")
print("✅ 0pentrons-robot-server has restart successfully!")

#Sometimes stopping the robot server can cause weird issues. If you run
    #os.system("systemctl start opentrons-robot-server") should respond with zero. If it doesn't you can do the following steps:
    # 1) Go to the "Running" tab
    # 2) Shut down all kernels
    # 3) open a new file with 
        #import os 
        #os.system("systemctl start opentrons-robot-server")
    # 4) If that doesn't work. Restart your OT-2, see if your Ot-2 connects.

<hr style="height: 2px; background-color: black; border: none;">

# OT2 commands

In [ ]:
#  Reset the the number of tips used
reset_tip_counters()
print("✅ Tip counters have been manually reset.")

In [ ]:
ctx.home()

In [ ]:
temp_mod_pos03.deactivate()

In [ ]:
temp_mod_pos09.deactivate()

In [ ]:
tc_mod.deactivate_block()

In [ ]:
p300.default_speed = 50
p300.pick_up_tip()

In [ ]:
p300.default_speed = 400
p300.drop_tip()

In [ ]:
p20.default_speed = 50
p20.pick_up_tip()

In [ ]:
p20.default_speed = 400
p20.drop_tip()

In [ ]:
p300.default_speed = 50
p300.move_to(aluminumblock_nest_24_500µL_pos09['A1'].top(z=0))
p300.default_speed = 400

In [ ]:
p300.default_speed = 50
p300.move_to(aluminumblock_nest_24_500µL_pos09['A1'].bottom(z=0.2))
p300.default_speed = 400

In [ ]:
p300.default_speed = 50
p300.move_to(aluminumblock_nest_24_500µL_pos09['B2'].top(z=-6))
p300.default_speed = 400

In [ ]:
p300.default_speed = 50
p300.move_to(aluminumblock_nest_24_500µL_pos09['B2'].bottom(z=0.2))
p300.default_speed = 400

In [ ]:
p20.default_speed = 50
p20.move_to(aluminumblock_nest_24_500µL_pos09['A1'].top(z=0))
p20.default_speed = 400

In [ ]:
p20.default_speed = 50
p20.move_to(aluminumblock_nest_24_500µL_pos09['A1'].bottom(z=0.2))
p20.default_speed = 400

In [ ]:
p300.default_speed = 50
p300.move_to(aluminumblock_nest_96_wellplate_100ul_pos03_from_fridge['A1'].top(z=0))
p300.default_speed = 400

In [ ]:
p300.default_speed = 50
p300.move_to(aluminumblock_nest_96_wellplate_100ul_pos03_from_fridge['A1'].bottom(z=0.2))
p300.default_speed = 400

In [ ]:
p20.default_speed = 50
p20.move_to(aluminumblock_nest_96_wellplate_100ul_pos03_from_fridge['A1'].top(z=0))
p20.default_speed = 400

In [ ]:
p20.default_speed = 50
p20.move_to(aluminumblock_nest_96_wellplate_100ul_pos03_from_fridge['A1'].bottom(z=0.2))
p20.default_speed = 400

In [ ]:
p20.default_speed = 50
p20.move_to(aluminumblock_nest_96_wellplate_100ul_pos03_from_fridge['H12'].bottom(z=bottom_z_microwell))
p20.default_speed = 400

In [ ]:
p20.default_speed = 50
p20.move_to(aluminumblock_nest_96_wellplate_100ul_pos03_from_fridge['A1'].bottom(z=0.1))
p20.default_speed = 400

In [ ]:
p300.default_speed = 50
p300.move_to(tc_plate['A1'].top(z=0))

In [ ]:
p20.default_speed = 50
p20.move_to(tc_plate['A1'].top(z=0))

In [ ]:
ws.safe(fridge.open_door)

In [ ]:
ws.safe(fridge.close_door)

In [ ]:
arm.mov_arm_coords_and_rail(*ws.pos_security)

In [ ]:
tc_mod.close_lid()

In [ ]:
tc_mod.open_lid()

In [ ]:
tc_mod.deactivate_lid()

In [ ]:
p300.default_speed = 50
p300.move_to(tipracks_p300_slot06['A1'].top(z=0))
p300.default_speed = 400

In [ ]:
p300.flow_rate.blow_out = 400
p300.blow_out()

# GET ERROT ID

In [ ]:
arm.dashboard.GetErrorID()()

# TEST p300 OFFSETS

In [ ]:
# Juste après ton patch, fait un petit test visible
loc0 = aluminumblock_nest_24_500µL_pos09['A1'].top(z=0)
print(f"Avant patch: {loc0.point}")           # point brut
loc1 = p300.move_to(aluminumblock_nest_24_500µL_pos09['A1'].top(z=0))  
# Tu ne verras pas le return de move_to, mais aucun erreur => patch appliqué

In [ ]:
start_time = time.time()  # start of the timer
time.sleep(5)  # wait 5 seconds: be ready to press the emergency stop button
arm.mov_arm_coords_and_rail(*ws.pos_security)  # in case it isn't yet
ws.do_action_slot(slot="fridge_slot4", griper_action=arm.close_gripper)
ws.do_action_slot(slot="temp_module_slot3", griper_action=arm.open_gripper)
step3_time = time.time() - start_time # Calculation of elapsed time

print(f"🦾 Step 3 completed in {step3_time:.2f} seconds.")

In [ ]:
start_time = time.time()  # start of the timer
time.sleep(5)  # wait 5 seconds: be ready to press the emergency stop button
arm.mov_arm_coords_and_rail(*ws.pos_security)  # in case it isn't yet
ws.do_action_slot(slot="temp_module_slot3", griper_action=arm.close_gripper)
ws.do_action_slot(slot="fridge_slot4", griper_action=arm.open_gripper)
step3_time = time.time() - start_time # Calculation of elapsed time

print(f"🦾 Step 3 completed in {step3_time:.2f} seconds.")